<a href="https://colab.research.google.com/github/chayne-wild/PCAP-CLIP-and-LLaVA/blob/main/PCAP_CLIP_and_LLaVA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Counter-Visuality in the Carceral Archive: Distant Viewing and Multimodal Interrogation of the Prison Creative Arts Project (PCAP)

**Author:** Chayne Wild (Alliance for Social, Political, Ethical, and Cultural Thought [ASPECT], Virginia Tech)  
**Repository:** [https://github.com/chayne-wild/PCAP-CLIP-and-LLaVA](https://github.com/chayne-wild/PCAP-CLIP-and-LLaVA)  
_# SPDX-License-Identifier: MIT_

---

### Overview & Sequential Pipeline Architecture
This notebook executes the complete end-to-end replication pipeline sequentially (`00` $\rightarrow$ `09`) in **Google Colaboratory** using the `rpy2` (`%%R`) bridge:
1. **Step 00 (`Python / R`):** Google Colab Environment Setup, Drive Mount, Conditional GPU Verification & `rpy2` R Binary Provisioning
2. **Step 01 (`Python Local/Terminal`):** Polite Omeka S REST API Metadata & Media URL Harvester (`data/pcap_master_metadata.csv`)
3. **Step 02 (`Python GPU`):** Ephemeral RAM Streaming & `CLIP ViT-L/14` Distant Viewing (`data/pcap_clip_embeddings.parquet`)
4. **Step 03 (`Python GPU`):** Stratified High-Signal Subsampling ($\tau \ge 0.90, N=1,200$) & 4-bit `LLaVA-1.5-7B` Interrogation (`data/pcap_llava_interrogations.jsonl`)
5. **Step 04 (`R %%R`):** Tidy Metadata Normalization, Disalienation Hypothesis Test & 768-D CLIP UMAP Projection
6. **Step 05 (`R %%R`):** LLaVA Tidy NLP, "Visual Plural Bypass" Filtering & STM `searchK` Diagnostics
7. **Step 06 (`R %%R`):** Deterministic $K = 25$ Spectral Structural Topic Model (STM) on Visual Motifs
8. **Step 07 (`R %%R`):** B-Spline Temporal Trajectories & Surge-Era Counter-Visuality OLS Regressions (1996–2024 vs. 2010–2024)
9. **Step 08 (`R %%R`):** Cross-Modal Network Topology (`stm::topicCorr` & `igraph` Multipartite Graph)
10. **Step 09 (`R %%R`):** Joint CLIP–LLaVA Exemplar Extraction & Qualitative Artwork Dossier

---
## Step 00: Colab Environment Setup (`00_colab_environment_setup.py`)
Mounts Google Drive, locks the working directory, creates the `data/` and `results/` hierarchy, conditionally installs Python GPU/vision libraries when a CUDA runtime is active, activates `rpy2.ipython` (`%%R`), and caches pre-compiled Ubuntu Linux binary R packages in Google Drive (`r_libs/`).

* **Runtime Switching Note:** Step 00 dynamically detects whether a CPU or GPU runtime is active. Re-run this cell whenever switching Colab runtimes between **GPU** (required for Steps 02–03) and **CPU** (recommended for Steps 04–09 to conserve compute units).

In [ ]:
# ==============================================================================
# scripts/00_colab_environment_setup.py (FAST BINARY & DRIVE-CACHED)
# Cultural Resistance: Counter-Visuality in the PCAP Archive
# ==============================================================================

import os
import sys
import subprocess
from pathlib import Path
from typing import List

# ------------------------------------------------------------------------------
# 1. GOOGLE DRIVE MOUNT & WORKING DIRECTORY LOCK
# ------------------------------------------------------------------------------
COLAB_ROOT = Path("/content/drive/MyDrive/colab_pcap_folder")
REQUIRED_DIRS = [
    COLAB_ROOT / "data",
    COLAB_ROOT / "scripts",
    COLAB_ROOT / "results" / "charts",
    COLAB_ROOT / "results" / "figures",
    COLAB_ROOT / "r_libs",  # Persistent R binary cache across Colab disconnects
]


def initialize_workspace(root_path: Path, subdirs: List[Path]) -> Path:
    """Mount Google Drive (if in Colab) and create the required directory tree."""
    print("=" * 78)
    print("[INFO] PHASE 0: Initializing PCAP Counter-Visuality Colab Environment")
    print("=" * 78)

    try:
        from google.colab import drive  # type: ignore
        if not Path("/content/drive/MyDrive").exists():
            print("[INFO] Mounting Google Drive at /content/drive ...")
            drive.mount("/content/drive", force_remount=False)
        else:
            print("[INFO] Google Drive is already mounted.")
    except ImportError:
        print("[WARN] Not running inside Google Colab. Using local fallback path.")
        root_path = Path.cwd() / "colab_pcap_folder"
        subdirs = [
            root_path / "data",
            root_path / "scripts",
            root_path / "results" / "charts",
            root_path / "results" / "figures",
            root_path / "r_libs",
        ]

    for directory in [root_path] + subdirs:
        directory.mkdir(parents=True, exist_ok=True)
        print(f"[INFO] Verified directory: {directory}")

    os.chdir(root_path)
    assert Path.cwd().resolve() == root_path.resolve(), (
        f"[ERROR] Failed to lock working directory to {root_path}"
    )
    print(f"[TDD PASS] Working directory locked to: {Path.cwd()}")
    return root_path


COLAB_ROOT = initialize_workspace(COLAB_ROOT, REQUIRED_DIRS)

# ------------------------------------------------------------------------------
# 2. CONDITIONAL PYTHON GPU INSTALL (SKIPS HEAVY GPU WHEELS ON CPU RUNTIMES)
# ------------------------------------------------------------------------------
import torch

HAS_GPU = torch.cuda.is_available()

if HAS_GPU:
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"\n[TDD PASS] GPU Runtime Active: {device_name} ({vram_gb:.2f} GB VRAM)")
    print("[INFO] Installing Python GPU & vision libraries for Steps 02-03...")
    GPU_PACKAGES = [
        "open_clip_torch>=2.24.0",
        "transformers>=4.40.0,<5.0.0",
        "bitsandbytes>=0.43.0",
        "accelerate>=0.30.0",
    ]
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + GPU_PACKAGES
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(f"[ERROR] pip installation failed:\n{res.stderr}")
    print("[TDD PASS] Python GPU libraries installed successfully.")
else:
    print(
        "\n[INFO] CPU Runtime Detected! Skipping GPU-only Python wheels "
        "(open_clip, bitsandbytes) to save startup time.\n"
        "       -> CPU is ready for Step 01 (API Harvest) and Steps 04-09 (R/STM)."
    )

# ------------------------------------------------------------------------------
# 3. FAST UBUNTU APT BINARY PRE-INSTALL FOR HEAVY C++ R PACKAGES
# ------------------------------------------------------------------------------
APT_R_BINARIES = [
    "r-cran-tidyverse",
    "r-cran-arrow",
    "r-cran-jsonlite",
    "r-cran-uwot",
    "r-cran-cluster",
    "r-cran-tidytext",
    "r-cran-tm",
    "r-cran-snowballc",
    "r-cran-stm",
    "r-cran-igraph",
    "r-cran-broom",
    "r-cran-patchwork",
    "r-cran-scales",
    "r-cran-matrix",
]

if Path("/usr/bin/apt-get").exists():
    print("\n[INFO] Installing pre-compiled Ubuntu .deb binaries for heavy C++ R packages...")
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    # Install each available apt binary quietly (ignores any not in current PPA)
    for apt_pkg in APT_R_BINARIES:
        subprocess.run(
            ["apt-get", "install", "-y", "-qq", apt_pkg],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
        )
    print("[TDD PASS] Ubuntu system R binary check complete.")

# ------------------------------------------------------------------------------
# 4. ACTIVATE RPY2 RMAGIC FOR GOOGLE COLAB
# ------------------------------------------------------------------------------
try:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.run_line_magic("reload_ext", "rpy2.ipython")
        print("[TDD PASS] rpy2.ipython extension (%R / %%R) activated.")
except Exception as exc:
    print(f"[WARN] Could not auto-activate rpy2 magic via IPython: {exc}")

# ------------------------------------------------------------------------------
# 5. R BINARY SETUP (POSIT HTTPUserAgent + DRIVE CACHE + dependencies = NA)
# ------------------------------------------------------------------------------
R_SETUP_SCRIPT = """
options(warn = 1)
cat("==============================================================================\\n")
cat("[INFO] Configuring Persistent Drive R Cache & Posit Linux Binary Repo...\\n")
cat("==============================================================================\\n")

# 1. Mount persistent R library cache on Google Drive so packages survive disconnects
drive_r_lib <- "/content/drive/MyDrive/colab_pcap_folder/r_libs"
if (dir.exists(drive_r_lib)) {
  .libPaths(unique(c(drive_r_lib, .libPaths())))
  cat(sprintf("[INFO] Persistent Drive R library active: %s\\n", drive_r_lib))
}

# 2. CRITICAL: Set HTTPUserAgent so Posit Package Manager serves pre-compiled
#    Linux binaries instead of C++ source tarballs!
options(
  HTTPUserAgent = sprintf(
    "R/%s R (%s)",
    getRversion(),
    paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"])
  )
)

os_codename <- "jammy"
if (file.exists("/etc/os-release")) {
  os_lines <- readLines("/etc/os-release", warn = FALSE)
  code_line <- grep("^VERSION_CODENAME=", os_lines, value = TRUE)
  if (length(code_line) > 0) {
    os_codename <- sub("^VERSION_CODENAME=", "", gsub('"', '', code_line[1]))
  }
}
rspm_url <- sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", os_codename)
cat(sprintf("[INFO] Using Posit Binary Repo (%s) with binary User-Agent.\\n", os_codename))

options(
  repos = c(
    RSPM = rspm_url,
    CRAN = "https://cloud.r-project.org"
  ),
  Ncpus = parallel::detectCores()
)

required_r_pkgs <- c(
  "tidyverse",
  "arrow",
  "jsonlite",
  "uwot",
  "cluster",
  "tidytext",
  "textstem",
  "tm",
  "SnowballC",
  "stm",
  "igraph",
  "broom",
  "modelsummary",
  "gt",
  "patchwork",
  "scales",
  "Matrix"
)

missing_pkgs <- required_r_pkgs[!vapply(required_r_pkgs, requireNamespace, logical(1), quietly = TRUE)]

if (length(missing_pkgs) > 0) {
  cat(sprintf("[INFO] Installing %d remaining R packages from binary repo: %s\\n",
              length(missing_pkgs), paste(missing_pkgs, collapse = ", ")))
  # CRITICAL: Use dependencies = NA (Depends/Imports/LinkingTo only), NOT TRUE!
  install.packages(missing_pkgs, lib = .libPaths()[1], dependencies = NA, quiet = TRUE)
} else {
  cat("[INFO] All 15 required R packages are already cached and ready!\\n")
}

failed_pkgs <- character(0)
for (pkg in required_r_pkgs) {
  ok <- suppressPackageStartupMessages(
    require(pkg, character.only = TRUE, quietly = TRUE)
  )
  if (!ok) {
    failed_pkgs <- c(failed_pkgs, pkg)
  }
}

if (length(failed_pkgs) > 0) {
  stop(sprintf("[ERROR] The following R packages failed to load: %s",
               paste(failed_pkgs, collapse = ", ")))
}

cat("[TDD PASS] All 15 R packages installed and verified for Phase 2 (Scripts 04-09).\\n")
"""

if __name__ == "__main__":
    print("\n[INFO] Executing R environment provisioning via rpy2...")
    import rpy2.robjects as robjects
    robjects.r(R_SETUP_SCRIPT)

    if not HAS_GPU:
        print("\n" + "!" * 78)
        print("[REMINDER FOR STEPS 02 & 03 — GPU REQUIRED]")
        print("  You are currently on a CPU runtime (Python GPU wheels were skipped).")
        print("  After Step 01 finishes, switch your Colab runtime to a GPU (T4/L4/A100)")
        print("  and RE-RUN THIS STEP 00 CELL before running Steps 02 and 03!")
        print("  (Your R packages are cached in Drive r_libs/, so re-running will be fast.)")
        print("!" * 78)

[INFO] PHASE 0: Initializing PCAP Counter-Visuality Colab Environment
[INFO] Google Drive is already mounted.
[INFO] Verified directory: /content/drive/MyDrive/colab_pcap_folder
[INFO] Verified directory: /content/drive/MyDrive/colab_pcap_folder/data
[INFO] Verified directory: /content/drive/MyDrive/colab_pcap_folder/scripts
[INFO] Verified directory: /content/drive/MyDrive/colab_pcap_folder/results/charts
[INFO] Verified directory: /content/drive/MyDrive/colab_pcap_folder/results/figures
[INFO] Verified directory: /content/drive/MyDrive/colab_pcap_folder/r_libs
[TDD PASS] Working directory locked to: /content/drive/MyDrive/colab_pcap_folder

[INFO] CPU Runtime Detected! Skipping GPU-only Python wheels (open_clip, bitsandbytes) to save startup time.
       -> CPU is ready for Step 01 (API Harvest) and Steps 04-09 (R/STM).

[INFO] Installing pre-compiled Ubuntu .deb binaries for heavy C++ R packages...
[TDD PASS] Ubuntu system R binary check complete.
[TDD PASS] rpy2.ipython extension (

---
## Step 01: Harvest PCAP Omeka S REST API (`01_harvest_pcap_omeka_api.py`)
Queries the University of Michigan Carceral State Project / PCAP Omeka S REST API (`/api/items` and `/api/media`, `site_id=17`) with polite rate-limiting (`1.5–3.0s`), academic identification headers (`User-Agent` and RFC 9110 `From`), and incremental checkpointing. Extracts structured Dublin Core metadata, filters out non-artwork institutional records, and indexes direct image URLs without downloading binary files ($13,546$ valid artworks indexed; $7$ non-artwork records excluded).

### Local Terminal Execution Protocol (Cloudflare WAF Bypass)
Because the University of Michigan's Cloudflare WAF challenges automated JSON requests originating from Google Cloud data-center IPs on `/api/items` (while permitting direct image file requests in Steps 02–03), Step 01 is designed to run from a local terminal using zero external dependencies (`urllib.request`) and write directly to the mounted Google Drive folder:

1. **Share Google Drive Folder with Linux (0 MB Local Disk Space):** In the ChromeOS **Files** app, navigate to **Google Drive $\rightarrow$ My Drive**, right-click **`colab_pcap_folder`** (created in Step 00), and select **"Share with Linux"**.
2. **Navigate to the Shared Drive Directory:** Open the **Terminal (`penguin`)** app and run:
   ```bash
   cd /mnt/chromeos/GoogleDrive/MyDrive/colab_pcap_folder && pwd
   ```
3. **Execute the Harvester:** Copy and paste the bash block in the cell below into your terminal to write and run `scripts/01_harvest_pcap_omeka_api.py`.

**Outputs:** `data/pcap_master_metadata.csv`, `data/pcap_excluded_items.csv`

In [ ]:
cat << 'EOF' > scripts/01_harvest_pcap_omeka_api.py
# ==============================================================================
# scripts/01_harvest_pcap_omeka_api.py (FULL ARCHIVE HARVEST)
# Cultural Resistance: Counter-Visuality in the PCAP Archive
# PURPOSE: Queries the University of Michigan PCAP Omeka S REST API using ONLY
#          Python's built-in standard library (0 MB external dependencies).
#          Writes directly to Google Drive on Chromebook or Colab.
# ETHICS : Explicit Virginia Tech researcher identification headers
#          (User-Agent, RFC 9110 From, X-Research-Project) + 1.5-3.0s delay.
# OUTPUTS: data/pcap_master_metadata.csv
#          data/pcap_excluded_items.csv
# ==============================================================================

import os
import re
import csv
import math
import json
import time
import random
import urllib.request
import urllib.parse
import urllib.error
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# ------------------------------------------------------------------------------
# 1. AUTO-DETECT GOOGLE DRIVE PATH (CHROMEBOOK LINUX OR GOOGLE COLAB)
# ------------------------------------------------------------------------------
CHROMEBOOK_DRIVE = Path("/mnt/chromeos/GoogleDrive/MyDrive/colab_pcap_folder")
COLAB_DRIVE = Path("/content/drive/MyDrive/colab_pcap_folder")

if CHROMEBOOK_DRIVE.exists():
    PROJECT_ROOT = CHROMEBOOK_DRIVE
elif COLAB_DRIVE.exists():
    PROJECT_ROOT = COLAB_DRIVE
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

MASTER_CSV_PATH = DATA_DIR / "pcap_master_metadata.csv"
EXCLUDED_CSV_PATH = DATA_DIR / "pcap_excluded_items.csv"
CHECKPOINT_JSON_PATH = DATA_DIR / "checkpoint_01_omeka_harvest.json"

OMEKA_BASE_URL = "https://dcc.carceralstateproject.lsa.umich.edu"
API_ITEMS_ENDPOINT = f"{OMEKA_BASE_URL}/api/items"
API_MEDIA_ENDPOINT = f"{OMEKA_BASE_URL}/api/media"
API_SITES_ENDPOINT = f"{OMEKA_BASE_URL}/api/sites"

# Ethical academic identification (User-Agent + RFC 9110 'From' + X-Research-Project)
RESEARCH_BOT_ID = "PCAP_Virginia_Tech_Research_Bot/1.0 (chayne@vt.edu)"
PRIMARY_HEADERS = {
    "User-Agent": RESEARCH_BOT_ID,
    "From": "chayne@vt.edu",
    "X-Research-Project": RESEARCH_BOT_ID,
    "Accept": "application/json",
}
# Fallback if Cloudflare WAF regex challenges the word "Bot", still preserving
# full Virginia Tech & chayne@vt.edu identification in User-Agent, From, and X-Research-Project:
FALLBACK_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; CrOS x86_64 14541.0.0) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36 "
        "PCAP_Virginia_Tech_Research/1.0 (+mailto:chayne@vt.edu)"
    ),
    "From": "chayne@vt.edu",
    "X-Research-Project": RESEARCH_BOT_ID,
    "Accept": "application/json",
}
ACTIVE_HEADERS = dict(PRIMARY_HEADERS)

PER_PAGE = 100
DELAY_MIN_SEC = 1.5
DELAY_MAX_SEC = 3.0
MAX_RETRIES = 4

# Full harvest enabled (None = harvest all ~13,553 artworks across all pages)
MAX_PAGES_PILOT: Optional[int] = None

NON_ARTWORK_TITLE_PATTERNS = re.compile(
    r"\b(logo|sponsor|credits|guest\s*book|audio\s*tour|banner|header\s*image|oembed\s*media|box\s*url\s*test)\b",
    re.IGNORECASE,
)

MASTER_COLUMNS = [
    "item_id",
    "identifier",
    "title",
    "raw_date",
    "exhibition_source",
    "artist_name",
    "facility_raw",
    "medium_raw",
    "subject_tags",
    "visual_description",
    "artist_statement",
    "rights",
    "media_id",
    "image_url",
    "omeka_web_url",
]

EXCLUDED_COLUMNS = [
    "item_id",
    "title",
    "exclusion_reason",
    "omeka_api_url",
]


# ------------------------------------------------------------------------------
# 2. ZERO-DEPENDENCY HTTP JSON HELPER (BUILT-IN URLLIB)
# ------------------------------------------------------------------------------
def http_get_json(url: str, params: Optional[dict] = None, timeout: int = 35) -> Tuple[int, dict, object]:
    """Perform HTTP GET using Python's built-in urllib.request and return (status, headers, json_data)."""
    global ACTIVE_HEADERS
    full_url = url
    if params:
        query_str = urllib.parse.urlencode(params)
        full_url = f"{url}?{query_str}"

    req = urllib.request.Request(full_url, headers=ACTIVE_HEADERS)
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            status = resp.getcode()
            resp_headers = {k.lower(): v for k, v in resp.info().items()}
            raw_bytes = resp.read()
            data = json.loads(raw_bytes.decode("utf-8"))
            return status, resp_headers, data
    except urllib.error.HTTPError as e:
        if e.code == 403 and ACTIVE_HEADERS.get("User-Agent") == RESEARCH_BOT_ID:
            print(
                "[INFO] Cloudflare WAF challenged 'Bot' keyword; switching to "
                "User-Agent with '+mailto:chayne@vt.edu' & 'From: chayne@vt.edu'..."
            )
            ACTIVE_HEADERS = dict(FALLBACK_HEADERS)
            req2 = urllib.request.Request(full_url, headers=ACTIVE_HEADERS)
            with urllib.request.urlopen(req2, timeout=timeout) as resp:
                status = resp.getcode()
                resp_headers = {k.lower(): v for k, v in resp.info().items()}
                raw_bytes = resp.read()
                data = json.loads(raw_bytes.decode("utf-8"))
                return status, resp_headers, data
        raise


# ------------------------------------------------------------------------------
# 3. JSON-LD PARSING & MEDIA EXTRACTION
# ------------------------------------------------------------------------------
def extract_property_values(
    item_json: dict,
    term_keys: Optional[List[str]] = None,
    label_keywords: Optional[List[str]] = None,
) -> str:
    extracted: List[str] = []

    if term_keys:
        for key in term_keys:
            entries = item_json.get(key, [])
            if isinstance(entries, list):
                for entry in entries:
                    if isinstance(entry, dict):
                        val = (
                            entry.get("@value")
                            or entry.get("o:label")
                            or entry.get("display_title")
                            or entry.get("@id")
                        )
                        if val:
                            cleaned = re.sub(r"\s+", " ", str(val)).strip()
                            if cleaned:
                                extracted.append(cleaned)

    if label_keywords:
        allow_statement = any("statement" in kw.lower() for kw in label_keywords)
        for key, entries in item_json.items():
            if not isinstance(entries, list) or key.startswith("o:"):
                continue
            for entry in entries:
                if isinstance(entry, dict):
                    prop_label = str(entry.get("property_label", "")).strip().lower()
                    if "statement" in prop_label and not allow_statement:
                        continue
                    if any(kw.lower() in prop_label for kw in label_keywords):
                        val = (
                            entry.get("@value")
                            or entry.get("o:label")
                            or entry.get("display_title")
                        )
                        if val:
                            cleaned = re.sub(r"\s+", " ", str(val)).strip()
                            if cleaned:
                                extracted.append(cleaned)

    seen = set()
    unique_vals = []
    for val in extracted:
        if val not in seen:
            seen.add(val)
            unique_vals.append(val)

    return "; ".join(unique_vals)


def extract_artist_statement_field(item_json: dict, artwork_title: str) -> str:
    literals: List[str] = []
    for key, entries in item_json.items():
        if not isinstance(entries, list) or key.startswith("o:"):
            continue
        for entry in entries:
            if not isinstance(entry, dict):
                continue
            prop_label = str(entry.get("property_label", "")).strip().lower()
            is_stmt_key = key in (
                "bibo:content",
                "dcterms:abstract",
                "extracttext:extracted_text",
            )
            if is_stmt_key or "statement" in prop_label:
                val = entry.get("@value") or entry.get("o:label")
                if val:
                    cleaned = re.sub(r"\s+", " ", str(val)).strip()
                    if cleaned and cleaned.lower() != artwork_title.lower():
                        literals.append(cleaned)

    if literals:
        return "; ".join(dict.fromkeys(literals))
    return ""


def resolve_image_url_from_item(
    item_json: dict,
) -> Tuple[Optional[int], Optional[str], Optional[str]]:
    media_list = item_json.get("o:media", [])
    if not isinstance(media_list, list) or len(media_list) == 0:
        return None, None, "no_attached_media_object"

    first_media = media_list[0]
    media_id = first_media.get("o:id") if isinstance(first_media, dict) else None

    thumb_urls = item_json.get("thumbnail_display_urls", {})
    if isinstance(thumb_urls, dict):
        direct_url = thumb_urls.get("large") or thumb_urls.get("medium")
        if direct_url and any(
            direct_url.lower().endswith(ext)
            for ext in (".jpg", ".jpeg", ".png", ".webp")
        ):
            return media_id, direct_url, None

    if media_id is not None:
        time.sleep(random.uniform(0.35, 0.8))
        try:
            status, _, media_json = http_get_json(f"{API_MEDIA_ENDPOINT}/{media_id}", timeout=30)
            if status == 200 and isinstance(media_json, dict):
                media_type = str(media_json.get("o:media_type", "")).lower()
                if media_type and not media_type.startswith("image/"):
                    return media_id, None, f"non_image_media_type:{media_type}"

                t_urls = media_json.get("o:thumbnail_urls", {})
                img_url = (
                    (t_urls.get("large") if isinstance(t_urls, dict) else None)
                    or media_json.get("o:original_url")
                    or (t_urls.get("medium") if isinstance(t_urls, dict) else None)
                )
                if img_url:
                    return media_id, str(img_url), None
                return media_id, None, "media_record_missing_image_url"
        except Exception as exc:
            return media_id, None, f"media_api_error:{type(exc).__name__}"

    return media_id, None, "unresolved_image_url"


def parse_omeka_item_record(
    item_json: dict,
    site_slug: str = "pcap",
) -> Tuple[Optional[Dict[str, str]], Optional[Dict[str, str]]]:
    item_id = item_json.get("o:id")
    raw_title = (
        item_json.get("o:title")
        or extract_property_values(item_json, ["dcterms:title"], ["title"])
        or "Untitled"
    )
    title = re.sub(r"\s+", " ", str(raw_title)).strip()

    if NON_ARTWORK_TITLE_PATTERNS.search(title):
        return None, {
            "item_id": str(item_id),
            "title": title,
            "exclusion_reason": "institutional_logo_or_non_artwork_page",
            "omeka_api_url": f"{API_ITEMS_ENDPOINT}/{item_id}",
        }

    media_id, image_url, exclusion_reason = resolve_image_url_from_item(item_json)
    if exclusion_reason is not None or not image_url:
        return None, {
            "item_id": str(item_id),
            "title": title,
            "exclusion_reason": exclusion_reason or "missing_image_url",
            "omeka_api_url": f"{API_ITEMS_ENDPOINT}/{item_id}",
        }

    source_str = extract_property_values(
        item_json,
        ["dcterms:source", "dcterms:isPartOf", "dcterms:relation"],
        ["source", "exhibition", "part of", "collection"],
    )
    identifier_str = extract_property_values(
        item_json,
        ["dcterms:identifier"],
        ["log number", "identifier"],
    )
    medium_str = extract_property_values(
        item_json,
        ["dcterms:medium"],
        ["medium", "material"],
    )
    raw_date = extract_property_values(
        item_json,
        ["dcterms:date", "dcterms:created", "dcterms:issued"],
        ["date", "year"],
    )
    artist_name = (
        extract_property_values(
            item_json,
            ["dcterms:creator", "dcterms:contributor"],
            ["artist", "creator"],
        )
        or "Anonymous / Unlisted Artist"
    )
    facility_raw = extract_property_values(
        item_json,
        ["dcterms:spatial", "dcterms:coverage", "dcterms:publisher"],
        ["prison", "facility", "institution", "correctional"],
    )
    subject_tags = extract_property_values(
        item_json,
        ["dcterms:subject"],
        ["subject", "theme", "keyword", "gallery"],
    )
    visual_description = extract_property_values(
        item_json,
        ["dcterms:description"],
        ["description", "visual description"],
    )
    artist_statement = extract_artist_statement_field(item_json, artwork_title=title)
    rights = extract_property_values(
        item_json,
        ["dcterms:rights", "dcterms:rightsHolder"],
        ["rights", "copyright"],
    )

    omeka_web_url = f"{OMEKA_BASE_URL}/s/{site_slug}/item/{item_id}"

    master_row = {
        "item_id": str(item_id),
        "identifier": identifier_str,
        "title": title,
        "raw_date": raw_date,
        "exhibition_source": source_str,
        "artist_name": artist_name,
        "facility_raw": facility_raw,
        "medium_raw": medium_str,
        "subject_tags": subject_tags,
        "visual_description": visual_description,
        "artist_statement": artist_statement,
        "rights": rights,
        "media_id": str(media_id) if media_id else "",
        "image_url": image_url,
        "omeka_web_url": omeka_web_url,
    }
    return master_row, None


# ------------------------------------------------------------------------------
# 4. TDD SELF-TEST SUITE & PCAP SITE_ID DISCOVERY
# ------------------------------------------------------------------------------
def run_tdd_tests() -> None:
    print("[INFO] Running TDD unit tests for Omeka S JSON-LD parser...")
    mock_valid_item = {
        "o:id": 14892,
        "o:title": "Psychoanalysis",
        "o:media": [{"@id": f"{API_MEDIA_ENDPOINT}/15001", "o:id": 15001}],
        "thumbnail_display_urls": {
            "large": f"{OMEKA_BASE_URL}/files/large/sample_hash.jpg"
        },
        "dcterms:creator": [{"property_label": "Artist", "@value": "Van"}],
        "dcterms:medium": [{"property_label": "Medium", "@value": "Soap and ink"}],
        "dcterms:date": [{"property_label": "Date", "@value": "2021"}],
        "dcterms:source": [
            {
                "property_label": "Source",
                "@value": "26th Annual Exhibition of Art by Michigan Prisoners",
            }
        ],
        "dcterms:spatial": [
            {"property_label": "Prison", "@value": "Parnall Correctional Facility"}
        ],
        "bibo:content": [
            {"property_label": "Artist Statement", "@value": "Art is my survival."}
        ],
    }

    valid_row, excl_row = parse_omeka_item_record(mock_valid_item)
    assert valid_row is not None and excl_row is None
    assert valid_row["artist_name"] == "Van"
    assert valid_row["facility_raw"] == "Parnall Correctional Facility"
    assert valid_row["medium_raw"] == "Soap and ink"
    assert valid_row["artist_statement"] == "Art is my survival."
    print("[TDD PASS] Omeka S JSON-LD parser and exclusion gates verified.")


def discover_pcap_site_info() -> Tuple[Optional[int], str]:
    """
    Find the exact Omeka S site_id for the PCAP Visual Art Database so we only
    harvest PCAP artworks and exclude other DCC sub-sites / IT test items.
    """
    try:
        status, _, sites = http_get_json(API_SITES_ENDPOINT, timeout=25)
        if status == 200 and isinstance(sites, list):
            for s in sites:
                title = str(s.get("o:title", "")).lower()
                slug = str(s.get("o:slug", ""))
                if "visual art database" in title or slug.lower() == "pcap":
                    site_id = s.get("o:id")
                    _, hdrs, _ = http_get_json(
                        API_ITEMS_ENDPOINT,
                        params={"site_id": site_id, "per_page": 1},
                        timeout=20,
                    )
                    total_count = int(hdrs.get("omeka-s-total-results", "0"))
                    if total_count >= 5000:
                        print(
                            f"[INFO] Discovered Master PCAP Site: '{s.get('o:title')}' "
                            f"(site_id={site_id}, slug='{slug}', total_items={total_count})"
                        )
                        return int(site_id), slug
    except Exception:
        pass

    try:
        status, _, known_item = http_get_json(f"{API_ITEMS_ENDPOINT}/75605", timeout=25)
        if status == 200 and isinstance(known_item, dict):
            for s_ref in known_item.get("o:site", []):
                cand_id = s_ref.get("o:id")
                if cand_id:
                    _, hdrs, _ = http_get_json(
                        API_ITEMS_ENDPOINT,
                        params={"site_id": cand_id, "per_page": 1},
                        timeout=20,
                    )
                    total_count = int(hdrs.get("omeka-s-total-results", "0"))
                    if 10000 <= total_count <= 20000:
                        print(
                            f"[INFO] Locked onto PCAP Visual Art Database via Item 75605: "
                            f"site_id={cand_id} (total_items={total_count})"
                        )
                        return int(cand_id), "pcap"
    except Exception as exc:
        print(f"[WARN] Could not probe Item 75605 for site_id ({exc}).")

    return None, "pcap"


# ------------------------------------------------------------------------------
# 5. MAIN FAULT-TOLERANT HARVESTER
# ------------------------------------------------------------------------------
def harvest_pcap_omeka_archive() -> None:
    print("=" * 78)
    print(f"[INFO] Writing directly to : {DATA_DIR}")
    print(f"[INFO] Polite User-Agent   : {ACTIVE_HEADERS['User-Agent']}")
    print(f"[INFO] RFC 9110 From Header: {ACTIVE_HEADERS['From']}")
    print(f"[INFO] Rate-Limit Delay    : {DELAY_MIN_SEC}-{DELAY_MAX_SEC}s between pages")
    print("=" * 78)
    run_tdd_tests()

    site_id, site_slug = discover_pcap_site_info()

    start_page = 1
    seen_item_ids = set()
    if CHECKPOINT_JSON_PATH.exists() and MASTER_CSV_PATH.exists():
        try:
            with open(CHECKPOINT_JSON_PATH, "r", encoding="utf-8") as f:
                ckpt = json.load(f)
                start_page = int(ckpt.get("last_completed_page", 0)) + 1
            with open(MASTER_CSV_PATH, "r", encoding="utf-8") as f:
                for row in csv.DictReader(f):
                    seen_item_ids.add(row["item_id"])
            print(
                f"[INFO] Resuming from checkpoint at Page {start_page} "
                f"({len(seen_item_ids)} artworks already indexed)."
            )
        except Exception:
            start_page = 1

    if start_page == 1 or not MASTER_CSV_PATH.exists():
        with open(MASTER_CSV_PATH, "w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=MASTER_COLUMNS).writeheader()
    if start_page == 1 or not EXCLUDED_CSV_PATH.exists():
        with open(EXCLUDED_CSV_PATH, "w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=EXCLUDED_COLUMNS).writeheader()

    page = start_page
    total_master = len(seen_item_ids)
    total_excluded = 0
    total_pages_est = MAX_PAGES_PILOT

    print(f"[INFO] Starting polite Full Archive harvest from {API_ITEMS_ENDPOINT} ...")

    while True:
        if MAX_PAGES_PILOT is not None and page > MAX_PAGES_PILOT:
            print(f"\n[INFO] Reached MAX_PAGES_PILOT={MAX_PAGES_PILOT}. Stopping.")
            break

        params = {
            "page": page,
            "per_page": PER_PAGE,
            "sort_by": "id",
            "sort_order": "asc",
        }
        if site_id is not None:
            params["site_id"] = site_id

        items_batch = None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                status, hdrs, data = http_get_json(API_ITEMS_ENDPOINT, params=params, timeout=45)
                if status == 200 and isinstance(data, list):
                    items_batch = data
                    total_hdr = hdrs.get("omeka-s-total-results")
                    if total_hdr and total_pages_est is None:
                        total_pages_est = math.ceil(int(total_hdr) / PER_PAGE)
                    break
            except Exception as err:
                wait = (2 ** attempt) + random.uniform(1.0, 3.0)
                print(f"[WARN] Attempt {attempt}/{MAX_RETRIES} on page {page} failed ({err}). Retrying in {wait:.1f}s...")
                time.sleep(wait)

        if items_batch is None:
            raise RuntimeError(f"[ERROR] Failed to fetch page {page} after {MAX_RETRIES} retries.")

        if len(items_batch) == 0:
            print(f"\n[INFO] Reached final empty page at page={page}. Harvest complete.")
            break

        master_rows_to_write = []
        excluded_rows_to_write = []

        for item_json in items_batch:
            if not isinstance(item_json, dict) or "o:id" not in item_json:
                continue

            item_id_str = str(item_json.get("o:id", ""))
            if item_id_str in seen_item_ids:
                continue

            master_row, excluded_row = parse_omeka_item_record(item_json, site_slug=site_slug)
            if master_row is not None:
                master_rows_to_write.append(master_row)
                seen_item_ids.add(item_id_str)
                total_master += 1
            elif excluded_row is not None:
                excluded_rows_to_write.append(excluded_row)
                total_excluded += 1

        if master_rows_to_write:
            with open(MASTER_CSV_PATH, "a", newline="", encoding="utf-8") as f:
                csv.DictWriter(f, fieldnames=MASTER_COLUMNS).writerows(master_rows_to_write)

        if excluded_rows_to_write:
            with open(EXCLUDED_CSV_PATH, "a", newline="", encoding="utf-8") as f:
                csv.DictWriter(f, fieldnames=EXCLUDED_COLUMNS).writerows(excluded_rows_to_write)

        with open(CHECKPOINT_JSON_PATH, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "last_completed_page": page,
                    "total_master_indexed": total_master,
                    "total_excluded": total_excluded,
                    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                },
                f,
                indent=2,
            )

        denom = f"/{total_pages_est}" if total_pages_est else ""
        print(
            f"  -> [Page {page}{denom}] Indexed {len(master_rows_to_write)} artworks "
            f"(Running Total: {total_master} valid, {total_excluded} excluded)"
        )

        if len(items_batch) < PER_PAGE:
            print(f"\n[INFO] Final partial page ({len(items_batch)} items) processed.")
            break

        page += 1
        time.sleep(random.uniform(DELAY_MIN_SEC, DELAY_MAX_SEC))

    print("=" * 78)
    print("[TDD PASS] Script 01 Complete!")
    print(f"  -> Master Metadata CSV : {MASTER_CSV_PATH} ({total_master} artworks)")
    print(f"  -> Excluded Items CSV  : {EXCLUDED_CSV_PATH} ({total_excluded} records)")
    print("=" * 78)


if __name__ == "__main__":
    harvest_pcap_omeka_archive()
EOF
python3 scripts/01_harvest_pcap_omeka_api.py

---
## Step 02: Ephemeral RAM Streaming & CLIP Distant Viewing (`02_stream_clip_distant_viewing.py`)
Streams each PCAP artwork's image bytes into volatile RAM (`io.BytesIO`), computes L2-normalized 768-D `OpenCLIP ViT-L/14` visual embeddings and zero-shot cosine similarity scores across the 6-Axis Theoretical Prompt Matrix, and immediately purges image binaries from memory.

* **GPU Runtime Requirement:** Switch your Colab runtime to **T4 GPU** and re-run **Step 00** before executing Steps 02 and 03.
* **Open-Weights & Data Sovereignty (Hugging Face Hub):** Pulling open-weights checkpoints directly from the Hugging Face Hub into isolated volatile memory avoids transmitting incarcerated artists' works to commercial third-party APIs. Because `OpenCLIP` and `LLaVA-1.5-7B` are public open-weights models, any Colab `HF_TOKEN` informational warning can be safely ignored.
* **Contrastive Surveillance Orthogonalization:** `score_surveillance_biometric` applies Contrastive Concept Subtraction ($\alpha = 0.25$) to subtract close-up crying/reflective eye portraiture, isolating institutional and panoptic surveillance (`r = 0.737` with carceral containment).

**Output:** `data/pcap_clip_embeddings.parquet`

In [ ]:
# ==============================================================================
# scripts/02_stream_clip_distant_viewing.py
# Cultural Resistance: Counter-Visuality in the PCAP Archive
# PURPOSE: Streams each PCAP artwork's image bytes into volatile RAM (io.BytesIO),
#          computes L2-normalized 768-D CLIP (ViT-L/14) visual embeddings, and
#          calculates zero-shot cosine similarity scores against a 6-axis
#          Theoretical Prompt Matrix grounded in Fanon/Tosquelles (5.1),
#          Arnold & Tilton (5.2), and Mirzoeff/Browne (5.4).
#          Applies Contrastive Concept Subtraction (alpha = 0.25) on the
#          Panoptic Surveillance axis to orthogonalize institutional surveillance
#          from close-up portraits of crying/reflective eyes.
# ETHICS : ZERO PERSISTENT IMAGE STORAGE. Every image buffer is immediately
#          closed after the GPU forward pass.
# OUTPUT : data/pcap_clip_embeddings.parquet
# ==============================================================================

import io
import gc
import time
import random
import warnings
from pathlib import Path
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from PIL import Image, ImageFile
import torch
import open_clip
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")

# ------------------------------------------------------------------------------
# 1. PATHS, MODEL PARAMETERS & 6-AXIS THEORETICAL PROMPT MATRIX
# ------------------------------------------------------------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/colab_pcap_folder")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
MASTER_CSV_PATH = DATA_DIR / "pcap_master_metadata.csv"
CLIP_PARQUET_PATH = DATA_DIR / "pcap_clip_embeddings.parquet"

CLIP_MODEL_ARCH = "ViT-L-14"
CLIP_PRETRAINED = "openai"
EMBEDDING_DIM = 768
FLUSH_BATCH_SIZE = 100

USER_AGENT = "PCAP_Virginia_Tech_Research_Bot/1.0 (chayne@vt.edu)"
HEADERS = {"User-Agent": USER_AGENT}

# 6-Axis Theoretical Codebook (Ensemble prompts averaged in L2-normalized space)
THEORETICAL_PROMPT_MATRIX: Dict[str, List[str]] = {
    "score_surveillance_biometric": [
        "an artwork depicting a prison watchtower, security camera, searchlight beam, or panoptic surveillance",
        "being watched by correctional guards, disembodied watching eyes on a fence or sky, or cell observation window",
        "cctv camera lens, optical surveillance grid, retinal scan, or panopticon tower overlooking a prison",
    ],
    "score_carceral_containment": [
        "an artwork depicting steel prison bars, razor wire fence, locked cell door, and concrete walls",
        "claustrophobic prison cell interior, handcuffs, chains, shackles, and architectural enclosure",
        "barbed wire coils, solitary confinement box, and cold institutional prison corridor",
    ],
    "score_bodily_autonomy": [
        "a defiant self-portrait of a person returning the viewer's gaze with dignity and bodily sovereignty",
        "an expressive portrait of an assertive human figure reclaiming bodily autonomy and strength",
        "close-up human face looking directly forward with resilience, pride, and emotional depth",
    ],
    "score_tactile_disalienation": [
        "a richly textured handmade artwork with thick impasto, vibrant organic nature, trees, and open sky",
        "serene natural landscape, wildlife, flowing water, flowers, and warm tactile sensory vitality",
        "intricately crafted tactile surface celebrating nature, spiritual freedom, and open horizons",
    ],
    "score_digital_mediation": [
        "an artwork depicting a prison telephone behind plexiglass, video visitation monitor, or digital screen",
        "electronic cables, computer terminal, pixelated television screen, or telecommunications barrier",
        "talking through an institutional telephone handset or staring at a glowing digital kiosk interface",
    ],
    "score_state_bureaucracy": [
        "an artwork incorporating prisoner identification numbers, barcodes, mugshot placards, or state forms",
        "stamped bureaucratic numbers, inmate ID badge, legal documents, tally marks, and serial digits",
        "a human figure labeled with a barcode, prison identification number, or institutional paperwork",
    ],
}

# Contrastive Confounder for `score_surveillance_biometric` (Candidate B):
# Subtracts close-up crying/reflective eye portraiture (alpha = 0.25) so the axis
# isolates institutional/panoptic surveillance rather than subjective prisoner vision.
SURVEILLANCE_CONFOUNDER_PROMPTS: List[str] = [
    "a close-up pencil or charcoal drawing of a single sad human eye crying a tear",
    "close-up facial portrait focusing on eyelashes, eyebrows, and a reflective pupil",
]
ALPHA_CONFOUNDER_SUBTRACT = 0.25


# ------------------------------------------------------------------------------
# 2. MODEL INITIALIZATION & ZERO-SHOT TEXT ENCODING
# ------------------------------------------------------------------------------
def load_clip_and_encode_prompts(device: torch.device):
    """
    Load OpenCLIP ViT-L/14 and precompute the L2-normalized 768-D text prototype
    vectors for each of the 6 theoretical axes (applying contrastive confounder
    orthogonalization to `score_surveillance_biometric`).
    """
    print(f"[INFO] Loading OpenCLIP {CLIP_MODEL_ARCH} ({CLIP_PRETRAINED}) on {device}...")
    model, _, preprocess = open_clip.create_model_and_transforms(
        CLIP_MODEL_ARCH, pretrained=CLIP_PRETRAINED, device=device
    )
    model.eval()
    tokenizer = open_clip.get_tokenizer(CLIP_MODEL_ARCH)

    axis_names = list(THEORETICAL_PROMPT_MATRIX.keys())
    axis_embeddings = []

    with torch.no_grad():
        # Pre-encode confounder vector for contrastive surveillance axis
        conf_tokens = tokenizer(SURVEILLANCE_CONFOUNDER_PROMPTS).to(device)
        conf_feats = model.encode_text(conf_tokens).float()
        conf_feats = conf_feats / conf_feats.norm(dim=-1, keepdim=True)
        conf_mean = conf_feats.mean(dim=0, keepdim=True)
        conf_mean = conf_mean / conf_mean.norm(dim=-1, keepdim=True)

        for axis in axis_names:
            prompts = THEORETICAL_PROMPT_MATRIX[axis]
            tokens = tokenizer(prompts).to(device)
            text_feats = model.encode_text(tokens).float()
            text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)
            mean_feat = text_feats.mean(dim=0, keepdim=True)
            mean_feat = mean_feat / mean_feat.norm(dim=-1, keepdim=True)

            if axis == "score_surveillance_biometric":
                mean_feat = mean_feat - (ALPHA_CONFOUNDER_SUBTRACT * conf_mean)
                mean_feat = mean_feat / mean_feat.norm(dim=-1, keepdim=True)

            axis_embeddings.append(mean_feat)

    # Shape: (6, 768)
    prompt_matrix_tensor = torch.cat(axis_embeddings, dim=0)
    assert prompt_matrix_tensor.shape == (6, EMBEDDING_DIM), (
        f"[ERROR] Expected (6, {EMBEDDING_DIM}) prompt tensor, got {prompt_matrix_tensor.shape}"
    )
    print("[TDD PASS] 6-Axis Theoretical Prompt Matrix (with Contrastive Surveillance) encoded.")
    return model, preprocess, axis_names, prompt_matrix_tensor


# ------------------------------------------------------------------------------
# 3. EPHEMERAL RAM IMAGE STREAMING & FORWARD PASS
# ------------------------------------------------------------------------------
def stream_and_extract_single_artwork(
    image_url: str,
    session: requests.Session,
    model,
    preprocess,
    prompt_matrix_tensor: torch.Tensor,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Stream image bytes into an ephemeral io.BytesIO RAM buffer, compute the 768-D
    CLIP embedding and 6 cosine similarity scores, and immediately purge the
    image binary from memory.
    """
    buffer = None
    pil_image = None
    try:
        resp = session.get(image_url, headers=HEADERS, timeout=30)
        resp.raise_for_status()

        buffer = io.BytesIO(resp.content)
        pil_image = Image.open(buffer).convert("RGB")
        image_tensor = preprocess(pil_image).unsqueeze(0).to(device)

        with torch.no_grad():
            if device.type == "cuda":
                with torch.amp.autocast("cuda"):
                    img_feat = model.encode_image(image_tensor)
            else:
                img_feat = model.encode_image(image_tensor)

            img_feat = img_feat.float()
            img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)

            sim_scores = (img_feat @ prompt_matrix_tensor.T).squeeze(0)

            emb_np = img_feat.squeeze(0).cpu().numpy().astype(np.float32)
            scores_np = sim_scores.cpu().numpy().astype(np.float32)

        return emb_np, scores_np

    finally:
        if pil_image is not None:
            pil_image.close()
            del pil_image
        if buffer is not None:
            buffer.close()
            del buffer


# ------------------------------------------------------------------------------
# 4. TDD SELF-TEST ON SYNTHETIC IN-MEMORY TENSOR
# ------------------------------------------------------------------------------
def run_tdd_clip_verification(model, preprocess, prompt_matrix_tensor, device) -> None:
    """Verify forward pass dimensions and L2 normalization using an in-memory image."""
    test_buf = io.BytesIO()
    test_img = Image.new("RGB", (224, 224), color=(90, 120, 160))
    test_img.save(test_buf, format="JPEG")
    test_buf.seek(0)

    loaded_img = Image.open(test_buf).convert("RGB")
    img_tensor = preprocess(loaded_img).unsqueeze(0).to(device)

    with torch.no_grad():
        feat = model.encode_image(img_tensor).float()
        feat = feat / feat.norm(dim=-1, keepdim=True)
        scores = (feat @ prompt_matrix_tensor.T).squeeze(0)

    loaded_img.close()
    test_buf.close()
    del loaded_img, test_buf, test_img

    l2_norm = float(torch.norm(feat).item())
    assert abs(l2_norm - 1.0) < 1e-4, f"[ERROR] Embedding L2 norm={l2_norm} != 1.0"
    assert feat.shape == (1, EMBEDDING_DIM), f"[ERROR] Invalid shape {feat.shape}"
    assert scores.shape == (6,), f"[ERROR] Invalid score vector shape {scores.shape}"
    print(f"[TDD PASS] Ephemeral CLIP forward pass verified (L2 norm={l2_norm:.5f}).")


# ------------------------------------------------------------------------------
# 5. MAIN STREAMING ORCHESTRATOR WITH PARQUET CHECKPOINTING
# ------------------------------------------------------------------------------
def main() -> None:
    if not MASTER_CSV_PATH.exists():
        raise FileNotFoundError(
            f"[ERROR] Missing {MASTER_CSV_PATH}. Run Script 01 first!"
        )

    master_df = pd.read_csv(MASTER_CSV_PATH, dtype=str).fillna("")
    print(f"[INFO] Loaded {len(master_df)} artwork records from {MASTER_CSV_PATH}.")

    processed_ids = set()
    existing_df = None
    if CLIP_PARQUET_PATH.exists():
        try:
            existing_df = pd.read_parquet(CLIP_PARQUET_PATH)
            processed_ids = set(existing_df["item_id"].astype(str).tolist())
            print(
                f"[INFO] Resuming from existing Parquet checkpoint: "
                f"{len(processed_ids)} artworks already embedded."
            )
        except Exception as exc:
            print(f"[WARN] Could not read existing Parquet file ({exc}); starting fresh.")
            existing_df = None

    remaining_df = master_df[~master_df["item_id"].astype(str).isin(processed_ids)].copy()
    if len(remaining_df) == 0:
        print("[TDD PASS] All artworks in master CSV are already embedded in Parquet!")
        return

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, preprocess, axis_names, prompt_matrix_tensor = load_clip_and_encode_prompts(device)
    run_tdd_clip_verification(model, preprocess, prompt_matrix_tensor, device)

    emb_col_names = [f"clip_{i:03d}" for i in range(1, EMBEDDING_DIM + 1)]
    session = requests.Session()
    batch_records = []

    print(
        f"[INFO] Streaming {len(remaining_df)} remaining artworks into volatile RAM..."
    )
    pbar = tqdm(remaining_df.to_dict(orient="records"), desc="CLIP Distant Viewing")

    for row in pbar:
        item_id = str(row["item_id"])
        img_url = str(row["image_url"]).strip()
        if not img_url:
            continue

        try:
            emb_vec, score_vec = stream_and_extract_single_artwork(
                image_url=img_url,
                session=session,
                model=model,
                preprocess=preprocess,
                prompt_matrix_tensor=prompt_matrix_tensor,
                device=device,
            )

            out_record = dict(row)
            for ax_name, sc_val in zip(axis_names, score_vec):
                out_record[ax_name] = float(sc_val)
            for col_name, e_val in zip(emb_col_names, emb_vec):
                out_record[col_name] = float(e_val)

            batch_records.append(out_record)

        except Exception as err:
            print(f"\n[WARN] Skipping item_id={item_id} due to stream error: {err}")
            continue

        time.sleep(random.uniform(0.3, 0.8))

        if len(batch_records) >= FLUSH_BATCH_SIZE:
            batch_df = pd.DataFrame(batch_records)
            if existing_df is not None:
                combined_df = pd.concat([existing_df, batch_df], ignore_index=True)
            else:
                combined_df = batch_df
            combined_df.to_parquet(CLIP_PARQUET_PATH, index=False, engine="pyarrow")
            existing_df = combined_df
            batch_records.clear()
            gc.collect()

    if batch_records:
        batch_df = pd.DataFrame(batch_records)
        if existing_df is not None:
            combined_df = pd.concat([existing_df, batch_df], ignore_index=True)
        else:
            combined_df = batch_df
        combined_df.to_parquet(CLIP_PARQUET_PATH, index=False, engine="pyarrow")
        existing_df = combined_df
        batch_records.clear()
        gc.collect()

    assert CLIP_PARQUET_PATH.exists(), "[ERROR] Parquet output file was not created!"
    final_meta = pq.read_metadata(CLIP_PARQUET_PATH)
    print("=" * 78)
    print(f"[TDD PASS] Script 02 Complete!")
    print(f"  -> Output Parquet : {CLIP_PARQUET_PATH}")
    print(f"  -> Total Rows     : {final_meta.num_rows}")
    print(f"  -> Total Columns  : {final_meta.num_columns} (Metadata + 6 Scores + 768 Dims)")
    print("=" * 78)


if __name__ == "__main__":
    main()

---
## Step 03: Stratified High-Signal Subsampling & 4-Bit LLaVA Interrogation (`03_llava_stratified_interrogation.py`)
Selects a balanced 6-axis stratified high-signal subsample ($\tau \ge 0.90$; $150$ artworks per axis $\times 6$ axes $= 900$ high-signal + $300$ baseline control $= 1,200$ artworks), streams images into volatile RAM (`io.BytesIO`), and runs deterministic (`do_sample = False`) iconographic interrogation using 4-bit NormalFloat (`NF4`) quantized `llava-hf/llava-1.5-7b-hf`.

* **Hardware Democratization (`bitsandbytes` NF4):** 4-bit quantization compresses the 7B parameter Vision-Language Model from ~14 GB FP16 down to ~4.2 GB VRAM, enabling reproducible multimodal inference on a single Colab T4 GPU.
* **Runtime Transition After Step 03:** Once this cell completes, switch your Colab runtime back to **CPU** (and re-run **Step 00** once) to conserve GPU compute units during the R analytical pipeline (Steps 04–09).

**Output:** `data/pcap_llava_interrogations.jsonl`

In [ ]:
# ==============================================================================
# scripts/03_llava_stratified_interrogation.py
# Cultural Resistance: Counter-Visuality in the PCAP Archive
# PURPOSE: Selects a balanced 6-axis stratified high-signal subsample
#          and runs deterministic (`do_sample = False`) iconographic interrogation
#          using 4-bit NormalFloat (`NF4`) quantized llava-hf/llava-1.5-7b-hf
# OUTPUT : data/pcap_llava_interrogations.jsonl
# ==============================================================================

import io
import gc
import json
import time
import shutil
import random
import warnings
from pathlib import Path
from typing import Dict, List, Set, Tuple
import numpy as np
import pandas as pd
import requests
from PIL import Image, ImageFile
import torch
import open_clip
from transformers import AutoProcessor, BitsAndBytesConfig, LlavaForConditionalGeneration
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub")

# ------------------------------------------------------------------------------
# 1. CONFIGURATION, CONTRASTIVE SURVEILLANCE & OBJECTS+STUFF LLAVA PROMPT
# ------------------------------------------------------------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/colab_pcap_folder")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
CLIP_PARQUET_PATH = DATA_DIR / "pcap_clip_embeddings.parquet"
BACKUP_PARQUET_PATH = DATA_DIR / "pcap_clip_embeddings_v1_backup.parquet"
LLAVA_JSONL_PATH = DATA_DIR / "pcap_llava_interrogations.jsonl"

LLAVA_MODEL_ID = "llava-hf/llava-1.5-7b-hf"

# Set PILOT_MODE = True to test 6 top-ranked artworks; set False for full N=1,200 run
PILOT_MODE = False

TARGET_SAMPLE_SIZE = 1200
HIGH_SIGNAL_COUNT = 900    # Top 150 highest-scoring exemplars across all 6 axes
CONTROL_GROUP_COUNT = 300  # 300 random baseline control artworks (< 90th percentile)
QUANTILE_TAU = 0.90
RANDOM_SEED = 42

USER_AGENT = "PCAP_Virginia_Tech_Research_Bot/1.0 (chayne@vt.edu)"
HEADERS = {"User-Agent": USER_AGENT}

HIGH_SIGNAL_AXES = [
    "score_surveillance_biometric",
    "score_carceral_containment",
    "score_bodily_autonomy",
    "score_tactile_disalienation",
    "score_digital_mediation",
    "score_state_bureaucracy",
]

# Contrastive Panoptic Surveillance Definition (Candidate B)
SURVEILLANCE_TARGET_PROMPTS = [
    "an artwork depicting a prison watchtower, security camera, searchlight beam, or panoptic surveillance",
    "being watched by correctional guards, disembodied watching eyes on a fence or sky, or cell observation window",
    "cctv camera lens, optical surveillance grid, retinal scan, or panopticon tower overlooking a prison",
]
SURVEILLANCE_CONFOUNDER_PROMPTS = [
    "a close-up pencil or charcoal drawing of a single sad human eye crying a tear",
    "close-up facial portrait focusing on eyelashes, eyebrows, and a reflective pupil",
]
ALPHA_CONFOUNDER_SUBTRACT = 0.25

# Prompt: Prison archive context, asks for held objects & wall markings,
# and separates foreground "objects" from background "stuff" (Arnold & Tilton, 2023, pp. 103-105)
LLAVA_PROMPT_TEMPLATE = (
    "USER: <image>\n"
    "You are an art historian analyzing an artwork created by an incarcerated artist. "
    "Describe only what is visibly present in the work in three detailed paragraphs:\n"
    "1. Medium, colors, and formal texture: Describe the artistic medium, color palette, "
    "lighting, brushwork, line quality, and physical surface texture.\n"
    "2. Foreground figures and held objects: Describe any people or figures, their posture, "
    "facial expressions, the direction of their gaze, and anything worn on their bodies or "
    "held in their hands (such as chains, ropes, tools, cameras, phones, or weapons).\n"
    "3. Background stuff, architecture, and environment: Describe the surrounding setting, "
    "including any walls, fences, cell doors, windows, clocks, written text, numbers, symbols, "
    "or natural landscapes, water, trees, and open sky.\n"
    "ASSISTANT:"
)


# ------------------------------------------------------------------------------
# 2. SAFE IN-PLACE UPGRADE OF `score_surveillance_biometric` (CANDIDATE B)
# ------------------------------------------------------------------------------
def ensure_contrastive_surveillance_score() -> None:
    if BACKUP_PARQUET_PATH.exists():
        print(
            f"[INFO] Backup exists at {BACKUP_PARQUET_PATH.name}; "
            "Candidate B contrastive surveillance score is active."
        )
        return

    print("[INFO] Creating safe backup: data/pcap_clip_embeddings_v1_backup.parquet ...")
    shutil.copy2(CLIP_PARQUET_PATH, BACKUP_PARQUET_PATH)
    assert BACKUP_PARQUET_PATH.exists(), "[ERROR] Backup creation failed!"

    full_df = pd.read_parquet(CLIP_PARQUET_PATH)
    orig_rows, orig_cols = full_df.shape
    emb_cols = [f"clip_{i:03d}" for i in range(1, 769)]
    img_matrix = full_df[emb_cols].to_numpy(dtype=np.float32)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    clip_model, _, _ = open_clip.create_model_and_transforms(
        "ViT-L-14", pretrained="openai", device=device
    )
    clip_model.eval()
    tokenizer = open_clip.get_tokenizer("ViT-L-14")

    def _encode_ensemble(prompts: List[str]) -> np.ndarray:
        with torch.no_grad():
            tokens = tokenizer(prompts).to(device)
            feats = clip_model.encode_text(tokens).float()
            feats = feats / feats.norm(dim=-1, keepdim=True)
            mean_feat = feats.mean(dim=0, keepdim=True)
            mean_feat = mean_feat / mean_feat.norm(dim=-1, keepdim=True)
            return mean_feat.squeeze(0).cpu().numpy().astype(np.float32)

    vec_target = _encode_ensemble(SURVEILLANCE_TARGET_PROMPTS)
    vec_confounder = _encode_ensemble(SURVEILLANCE_CONFOUNDER_PROMPTS)
    vec_cand_b = vec_target - (ALPHA_CONFOUNDER_SUBTRACT * vec_confounder)
    vec_cand_b = (vec_cand_b / np.linalg.norm(vec_cand_b)).astype(np.float32)

    full_df["score_surveillance_biometric"] = (img_matrix @ vec_cand_b).astype(np.float32)
    assert full_df.shape == (orig_rows, orig_cols), "[ERROR] DataFrame shape mismatch!"
    full_df.to_parquet(CLIP_PARQUET_PATH, index=False, engine="pyarrow")
    print(f"[TDD PASS] Updated `score_surveillance_biometric` to Candidate B ({orig_rows} rows).")

    del clip_model, tokenizer, full_df, img_matrix
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------------------------------------
# 3. TOP-150 RANKED QUOTA SUBSAMPLING (`nlargest` PER AXIS + CONTROL)
# ------------------------------------------------------------------------------
def select_stratified_subsample(
    df: pd.DataFrame,
    target_n: int = TARGET_SAMPLE_SIZE,
    high_n: int = HIGH_SIGNAL_COUNT,
    control_n: int = CONTROL_GROUP_COUNT,
    tau: float = QUANTILE_TAU,
    seed: int = RANDOM_SEED,
) -> pd.DataFrame:
    """
    Selects the Top 150 highest Z-score artworks (`nlargest`) for each of the 6
    theoretical axes (900 high-signal total) + 300 random baseline controls.
    """
    thresholds = {ax: float(df[ax].quantile(tau)) for ax in HIGH_SIGNAL_AXES}
    print(f"[INFO] High-Signal Decile Thresholds (tau = {tau:.2f}):")
    for ax, val in thresholds.items():
        print(f"  -> {ax:<30}: {val:.4f}")

    z_scores = pd.DataFrame(index=df.index)
    for ax in HIGH_SIGNAL_AXES:
        std_val = float(df[ax].std()) or 1.0
        z_scores[ax] = (df[ax] - float(df[ax].mean())) / std_val

    high_mask = pd.Series(False, index=df.index)
    for ax in HIGH_SIGNAL_AXES:
        high_mask |= df[ax] >= thresholds[ax]

    high_pool = df[high_mask].copy()
    control_pool = df[~high_mask].copy()

    high_pool["sampling_stratum"] = (
        "high_signal_"
        + z_scores.loc[high_pool.index].idxmax(axis=1).str.replace("score_", "", regex=False)
    )
    high_pool["peak_z"] = z_scores.loc[high_pool.index].max(axis=1)
    control_pool["sampling_stratum"] = "baseline_control"
    control_pool["peak_z"] = 0.0

    if len(df) <= target_n:
        return pd.concat([high_pool, control_pool], ignore_index=True).drop(columns=["peak_z"])

    # Select the TOP 150 highest-scoring exemplars (`nlargest`) per stratum
    per_axis_quota = high_n // len(HIGH_SIGNAL_AXES)
    sampled_parts = []
    used_ids = set()

    for ax in HIGH_SIGNAL_AXES:
        stratum_name = "high_signal_" + ax.replace("score_", "")
        grp = high_pool[high_pool["sampling_stratum"] == stratum_name]
        n_take = min(per_axis_quota, len(grp))
        if n_take > 0:
            picked = grp.nlargest(n_take, "peak_z")
            sampled_parts.append(picked)
            used_ids.update(picked["item_id"].tolist())

    sampled_high = pd.concat(sampled_parts, ignore_index=True)

    if len(sampled_high) < high_n:
        shortfall = min(high_n - len(sampled_high), len(high_pool) - len(sampled_high))
        rem_pool = high_pool[~high_pool["item_id"].isin(used_ids)]
        extra_rows = rem_pool.nlargest(shortfall, "peak_z")
        sampled_high = pd.concat([sampled_high, extra_rows], ignore_index=True)

    actual_ctrl_n = min(control_n, len(control_pool), target_n - len(sampled_high))
    sampled_ctrl = control_pool.sample(n=actual_ctrl_n, random_state=seed)

    combined = (
        pd.concat([sampled_high, sampled_ctrl], ignore_index=True)
        .drop(columns=["peak_z"])
        .sample(frac=1.0, random_state=seed)
        .reset_index(drop=True)
    )

    print("\n[INFO] Final Stratified Sample Breakdown (Top-Ranked Exemplars + Control):")
    for st, cnt in combined["sampling_stratum"].value_counts().items():
        print(f"  -> {st:<40}: {cnt:4d} artworks")
    return combined


# ------------------------------------------------------------------------------
# 4. LOW-LEVEL HCI FORMAL METRICS & EPHEMERAL RAM LLAVA INTERROGATION
# ------------------------------------------------------------------------------
def compute_hci_formal_metrics(pil_image: Image.Image) -> Tuple[float, float, float]:
    """
    Computes 3 low-level formal color annotations in RAM following Arnold & Tilton
    (2023, Chapter 3, pp. 64-65, 93): mean intensity (brightness), mean chroma
    (saturation proxy), and warm-color pixel ratio.
    """
    arr = np.asarray(pil_image.resize((128, 128)), dtype=np.float32) / 255.0
    intensity = float(arr.mean())
    chroma = float((arr.max(axis=-1) - arr.min(axis=-1)).mean())
    # Warm pixels: Red channel exceeds Blue channel on non-grayscale pixels
    colorful_mask = (arr.max(axis=-1) - arr.min(axis=-1)) > 0.10
    if colorful_mask.sum() > 0:
        warm_ratio = float((arr[..., 0][colorful_mask] >= arr[..., 2][colorful_mask]).mean())
    else:
        warm_ratio = 0.5
    return round(intensity, 4), round(chroma, 4), round(warm_ratio, 4)


def load_quantized_llava(model_id: str):
    if not torch.cuda.is_available():
        raise RuntimeError("[ERROR] Script 03 requires a CUDA GPU runtime!")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    processor = AutoProcessor.from_pretrained(model_id)
    model = LlavaForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
        attn_implementation="sdpa",
    )
    model.eval()
    return model, processor


def interrogate_single_artwork_ram(
    image_url: str,
    session: requests.Session,
    model,
    processor,
) -> Tuple[str, float, float, float]:
    buffer = None
    pil_image = None
    inputs = None
    output_ids = None
    try:
        resp = session.get(image_url, headers=HEADERS, timeout=35)
        resp.raise_for_status()

        buffer = io.BytesIO(resp.content)
        pil_image = Image.open(buffer).convert("RGB")
        brightness, saturation, warm_ratio = compute_hci_formal_metrics(pil_image)

        inputs = processor(
            text=LLAVA_PROMPT_TEMPLATE,
            images=pil_image,
            return_tensors="pt",
        ).to(model.device, torch.float16)

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=325,
                do_sample=False,
                repetition_penalty=1.12,
            )

        decoded = processor.decode(output_ids[0], skip_special_tokens=True)
        transcript = (
            decoded.split("ASSISTANT:", 1)[-1].strip()
            if "ASSISTANT:" in decoded
            else decoded.strip()
        )
        return transcript, brightness, saturation, warm_ratio

    finally:
        if inputs is not None:
            del inputs
        if output_ids is not None:
            del output_ids
        if pil_image is not None:
            pil_image.close()
            del pil_image
        if buffer is not None:
            buffer.close()
            del buffer


# ------------------------------------------------------------------------------
# 5. MAIN EXECUTION LOOP
# ------------------------------------------------------------------------------
def main() -> None:
    ensure_contrastive_surveillance_score()

    meta_cols = [
        "item_id", "title", "raw_date", "exhibition_source",
        "medium_raw", "image_url", "omeka_web_url",
    ] + HIGH_SIGNAL_AXES

    df = pd.read_parquet(CLIP_PARQUET_PATH, columns=meta_cols)
    sample_df = select_stratified_subsample(df)

    model, processor = load_quantized_llava(LLAVA_MODEL_ID)
    session = requests.Session()

    if PILOT_MODE:
        print("\n" + "=" * 78)
        print("[PILOT MODE ACTIVE] Testing 6 top-ranked artworks (1 per axis)...")
        print("=" * 78)
        pilot_rows = (
            sample_df[sample_df["sampling_stratum"] != "baseline_control"]
            .groupby("sampling_stratum")
            .first()
            .reset_index()
        )
        for _, row in pilot_rows.iterrows():
            transcript, b_val, s_val, w_val = interrogate_single_artwork_ram(
                image_url=str(row["image_url"]).strip(),
                session=session,
                model=model,
                processor=processor,
            )
            print(f"\n>>> STRATUM : {row['sampling_stratum']}")
            print(f">>> TITLE   : {row['title']} (ID: {row['item_id']}) | Medium: {row['medium_raw']}")
            print(f">>> FORMAL  : Brightness={b_val:.3f}, Chroma={s_val:.3f}, WarmRatio={w_val:.3f}")
            print(f">>> URL     : {row['omeka_web_url']}")
            print(f">>> LLAVA   :\n{transcript}\n" + "-" * 78)
        print("[PILOT PASS] Flip PILOT_MODE = False (Line 41) for the full N=1,200 run.")
        return

    completed_ids: Set[str] = set()
    if LLAVA_JSONL_PATH.exists():
        with open(LLAVA_JSONL_PATH, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    try:
                        rec = json.loads(line)
                        if rec.get("item_id") and rec.get("llava_description"):
                            completed_ids.add(str(rec["item_id"]))
                    except json.JSONDecodeError:
                        continue

    pending_df = sample_df[~sample_df["item_id"].astype(str).isin(completed_ids)].copy()
    if len(pending_df) == 0:
        print("[TDD PASS] All 1,200 stratified artworks have been interrogated!")
        return

    with open(LLAVA_JSONL_PATH, "a", encoding="utf-8") as jsonl_file:
        pbar = tqdm(pending_df.to_dict(orient="records"), desc="LLaVA Visual Interrogation")
        for idx, row in enumerate(pbar, start=1):
            item_id = str(row["item_id"])
            img_url = str(row["image_url"]).strip()
            if not img_url:
                continue

            try:
                transcript, b_val, s_val, w_val = interrogate_single_artwork_ram(
                    image_url=img_url,
                    session=session,
                    model=model,
                    processor=processor,
                )
                record = {
                    "item_id": item_id,
                    "title": str(row.get("title", "")),
                    "sampling_stratum": str(row.get("sampling_stratum", "")),
                    "score_surveillance_biometric": float(row["score_surveillance_biometric"]),
                    "score_state_bureaucracy": float(row["score_state_bureaucracy"]),
                    "score_digital_mediation": float(row["score_digital_mediation"]),
                    "score_bodily_autonomy": float(row["score_bodily_autonomy"]),
                    "score_tactile_disalienation": float(row["score_tactile_disalienation"]),
                    "score_carceral_containment": float(row["score_carceral_containment"]),
                    "formal_brightness": b_val,
                    "formal_saturation": s_val,
                    "formal_warm_ratio": w_val,
                    "llava_model": LLAVA_MODEL_ID,
                    "llava_description": transcript,
                    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                }
                jsonl_file.write(json.dumps(record, ensure_ascii=False) + "\n")
                jsonl_file.flush()
            except Exception as err:
                print(f"\n[WARN] LLaVA failed on item_id={item_id}: {err}")
                continue

            if idx % 25 == 0:
                gc.collect()
                torch.cuda.empty_cache()
            time.sleep(random.uniform(0.3, 0.7))

    print(f"[TDD PASS] Script 03 Complete! Saved to {LLAVA_JSONL_PATH}")


if __name__ == "__main__":
    main()

[INFO] Backup exists at pcap_clip_embeddings_v1_backup.parquet; Candidate B contrastive surveillance score is active.
[INFO] High-Signal Decile Thresholds (tau = 0.90):
  -> score_surveillance_biometric  : 0.1724
  -> score_carceral_containment    : 0.1942
  -> score_bodily_autonomy         : 0.1863
  -> score_tactile_disalienation   : 0.1913
  -> score_digital_mediation       : 0.1639
  -> score_state_bureaucracy       : 0.1890

[INFO] Final Stratified Sample Breakdown (Top-Ranked Exemplars + Control):
  -> baseline_control                        :  300 artworks
  -> high_signal_state_bureaucracy           :  150 artworks
  -> high_signal_surveillance_biometric      :  150 artworks
  -> high_signal_bodily_autonomy             :  150 artworks
  -> high_signal_carceral_containment        :  150 artworks
  -> high_signal_tactile_disalienation       :  150 artworks
  -> high_signal_digital_mediation           :  150 artworks


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


... (11 outputs left out)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

LLaVA Visual Interrogation:   0%|          | 0/974 [00:00<?, ?it/s]

[TDD PASS] Script 03 Complete! Saved to /content/drive/MyDrive/colab_pcap_folder/data/pcap_llava_interrogations.jsonl


---
## Step 04: Tidy Metadata Normalization, Disalienation Test & CLIP UMAP (`04_tidy_metadata_and_umap.R`)
Ingests `data/pcap_clip_embeddings.parquet` natively in R via `arrow::read_parquet()`, normalizes exhibition years, MDOC security levels, and the 4-part Carceral Materiality Taxonomy (Section 5.1), runs `uwot::umap` + K-means clustering on the 768-D CLIP embedding space (Section 5.2), and generates Figures 01 & 02 and Tables 01 & 02.

* **CPU Runtime Note:** Steps 04–09 execute entirely in R on CPU (if you switched from GPU to CPU after Step 03, re-run **Step 00** once to mount Drive and load `rpy2`).

**Outputs:** `data/pcap_tidy_corpus.rds`, `results/charts/table01_pcap_medium_taxonomy.csv`, `results/charts/table02_cluster_prompt_profiles.csv`, `results/figures/fig01_disalienation_by_medium.png`, `results/figures/fig02_umap_visual_clusters.png`

In [ ]:
%%R
# ==============================================================================
# scripts/04_tidy_metadata_and_umap.R
# Cultural Resistance: Counter-Visuality in the PCAP Archive
# PURPOSE: Phase 2 R/Tidyverse Entrypoint. Ingests data/pcap_clip_embeddings.parquet
#          natively via arrow::read_parquet(). Cleans exhibition years, parses
#          Michigan DOC security levels, classifies raw medium strings into the
#          4-part carceral materiality taxonomy (Section 5.1), runs UMAP manifold
#          projection (uwot::umap) and K-means clustering on the 768-D CLIP space
#          (Section 5.2), and generates Figures 01 & 02 and Tables 01 & 02.
# OUTPUTS: data/pcap_tidy_corpus.rds
#          results/charts/table01_pcap_medium_taxonomy.csv
#          results/charts/table02_cluster_prompt_profiles.csv
#          results/figures/fig01_disalienation_by_medium.png
#          results/figures/fig02_umap_visual_clusters.png
# ==============================================================================

suppressPackageStartupMessages({
  library(tidyverse)
  library(arrow)
  library(uwot)
  library(cluster)
  library(patchwork)
  library(scales)
})

cat("==============================================================================\n")
cat("[INFO] SCRIPT 04: Tidy Metadata Normalization, Disalienation Test & CLIP UMAP\n")
cat("==============================================================================\n")

# ------------------------------------------------------------------------------
# 1. DIRECTORY & INPUT VERIFICATION
# ------------------------------------------------------------------------------
if (dir.exists("/content/drive/MyDrive/colab_pcap_folder")) {
  setwd("/content/drive/MyDrive/colab_pcap_folder")
}

parquet_path <- "data/pcap_clip_embeddings.parquet"
if (!file.exists(parquet_path)) {
  stop(sprintf("[ERROR] Missing input file: %s. Run Python Scripts 01-02 first!", parquet_path))
}

dir.create("data", showWarnings = FALSE, recursive = TRUE)
dir.create("results/charts", showWarnings = FALSE, recursive = TRUE)
dir.create("results/figures", showWarnings = FALSE, recursive = TRUE)

raw_clip_df <- arrow::read_parquet(parquet_path) %>% as_tibble()
cat(sprintf("[INFO] Ingested %d artworks and %d columns from %s.\n",
            nrow(raw_clip_df), ncol(raw_clip_df), parquet_path))

# ------------------------------------------------------------------------------
# 2. METADATA NORMALIZATION FUNCTIONS (YEAR, MEDIUM, SECURITY LEVEL)
# ------------------------------------------------------------------------------
# 2A. Exhibition Year Parser: Extracts 4-digit year (1996-2026) from raw_date,
#     or computes (1995 + N) from "Nth Annual Exhibition" in exhibition_source,
#     or parses the "-YY" suffix on PCAP Exhibition Log Numbers.
parse_exhibition_year <- function(raw_date, exhibition_source, identifier) {
  yr_direct <- str_extract(coalesce(raw_date, ""), "\\b(199[6-9]|20[0-2][0-9])\\b") %>%
    as.integer()

  ex_num <- str_match(
    tolower(coalesce(exhibition_source, "")),
    "\\b([0-9]{1,2})(?:st|nd|rd|th)\\s+annual\\s+exhibition"
  )[, 2] %>% as.integer()
  yr_from_ex <- ifelse(!is.na(ex_num) & ex_num >= 1 & ex_num <= 31, 1995L + ex_num, NA_integer_)

  log_yy <- str_match(coalesce(identifier, ""), "-[0-9]{2}$")[, 1] %>%
    str_remove("-") %>%
    as.integer()
  yr_from_log <- case_when(
    !is.na(log_yy) & log_yy >= 96 & log_yy <= 99 ~ 1900L + log_yy,
    !is.na(log_yy) & log_yy >= 0  & log_yy <= 26 ~ 2000L + log_yy,
    TRUE ~ NA_integer_
  )

  clean_yr <- coalesce(yr_direct, yr_from_ex, yr_from_log)
  clean_yr <- ifelse(clean_yr < 1996L | clean_yr > 2026L, NA_integer_, clean_yr)
  return(clean_yr)
}

# 2B. Material Medium Taxonomy (Section 5.1: Fanon/Tosquelles & Art Brut)
classify_carceral_medium <- function(medium_raw) {
  m_low <- tolower(coalesce(medium_raw, ""))

  case_when(
    # 1. Contraband / Found / Salvaged Carceral Substrates
    str_detect(
      m_low,
      paste(
        c("soap", "toilet paper", "tissue", "state form", "institutional form",
          "m&m", "candy wrapper", "wrapper", "cardboard", "corrugated",
          "matchstick", "match", "toothpaste", "coffee", "tea", "kool-aid",
          "bedsheet", "sheet", "pillowcase", "handkerchief", "sock", "jacket",
          "uniform", "deconstructed", "envelope", "newspaper", "book page",
          "floor wax", "deodorant", "salvaged", "found", "contraband",
          "bread", "foil", "playing card", "dried flower", "ash", "styrofoam"),
        collapse = "|"
      )
    ) ~ "Contraband/Found Carceral Material",

    # 2. Mixed Media & Sculptural / 3D Assemblage
    str_detect(
      m_low,
      paste(
        c("mixed media", "mixed", "collage", "sculpture", "ceramic", "clay",
          "wood", "carving", "assemblage", "papier", "paper mache", "origami",
          "3d", "diorama", "textile", "yarn", "thread", "bead", "mod podge"),
        collapse = "|"
      )
    ) ~ "Mixed/Sculptural",

    # 3. Graphite / Ink / Dry Drawing Media on Paper
    str_detect(
      m_low,
      paste(
        c("ballpoint", "pen", "ink", "graphite", "pencil", "charcoal",
          "colored pencil", "marker", "sharpie", "crayon", "chalk"),
        collapse = "|"
      )
    ) & !str_detect(m_low, "acrylic|oil|watercolor|gouache|tempera|canvas") ~ "Graphite/Ink on Paper",

    # 4. Standard Studio Paint & Canvas / Pastel
    str_detect(
      m_low,
      paste(
        c("acrylic", "oil", "watercolor", "gouache", "pastel", "tempera",
          "paint", "canvas"),
        collapse = "|"
      )
    ) ~ "Standard Paint/Canvas",

    # Fallback default for unspecified drawing/painting records
    TRUE ~ "Standard Paint/Canvas"
  )
}

# 2C. Michigan DOC Security Level Parser
parse_security_level <- function(facility_raw) {
  f_low <- tolower(coalesce(facility_raw, ""))
  case_when(
    f_low == "" ~ "Unspecified MDOC Facility",
    str_detect(f_low, "max|marquette|ionia|baraga|st\\. louis|algonac|standish|huron valley|reformatory") ~
      "Level IV-V (High / Maximum)",
    str_detect(f_low, "macomb|chippewa|kinross|muskegon|saginaw|carson city|bellamy|gus harrison|oaks|thumb|handlon|lakeland|g\\. robert cotton") ~
      "Level II-IV (Multi-Level Security)",
    str_detect(f_low, "parnall|cooper street|central michigan|earnest c\\. brooks|newberry|ojibway|camp") ~
      "Level I-II (Minimum / Low-Medium)",
    TRUE ~ "Level II-IV (Multi-Level Security)"
  )
}

# TDD Gate: Verify medium & year classifiers on edge cases before corpus application
stopifnot(
   classify_carceral_medium("Carved Ivory soap and toilet paper with ballpoint pen") ==
    "Contraband/Found Carceral Material",
  classify_carceral_medium("deconstructed and reconstructed jackets: fabric, acrylic paint and ink") ==
    "Contraband/Found Carceral Material",
  classify_carceral_medium("Ballpoint pen and graphite pencil on paper") ==
    "Graphite/Ink on Paper",
  classify_carceral_medium("Acrylic and oil on canvas") ==
    "Standard Paint/Canvas",
  parse_exhibition_year("", "26th Annual Exhibition of Art by Michigan Prisoners", "") == 2021L
)
cat("[TDD PASS] Metadata normalization helper functions verified.\n")

# ------------------------------------------------------------------------------
# 3. APPLY TIDY NORMALIZATION & COMPUTE DISALIENATION METRICS (SECTION 5.1)
# ------------------------------------------------------------------------------
metadata_df <- raw_clip_df %>%
  select(-starts_with("clip_")) %>%
  mutate(
    item_id = as.character(item_id),
    clean_year = parse_exhibition_year(raw_date, exhibition_source, identifier),
    year_imputed = is.na(clean_year),
    # Separate imputed year strictly for STM prevalence spline priors (Steps 05-06)
    clean_year_stm = coalesce(
      clean_year,
      median(clean_year, na.rm = TRUE) %>% as.integer(),
      2012L
    ),
    # Half-decade temporal buckets (Arnold & Tilton, 2023, Table 3.1, p. 71)
    half_decade = case_when(
      is.na(clean_year)   ~ "Undated Record",
      clean_year <= 1999L ~ "1996-1999",
      clean_year <= 2004L ~ "2000-2004",
      clean_year <= 2009L ~ "2005-2009",
      clean_year <= 2014L ~ "2010-2014",
      clean_year <= 2019L ~ "2015-2019",
      TRUE                ~ "2020-2024"
    ),
    era_window = case_when(
      is.na(clean_year)   ~ "Undated Record",
      clean_year >= 2010L ~ "Carceral-Digital Turn (2010-2024)",
      TRUE                ~ "Pre-Digital Era (1996-2009)"
    ),
    medium_category = factor(
      classify_carceral_medium(medium_raw),
      levels = c(
        "Contraband/Found Carceral Material",
        "Mixed/Sculptural",
        "Graphite/Ink on Paper",
        "Standard Paint/Canvas"
      )
    ),
    security_level = parse_security_level(facility_raw),
    disalienation_index = as.numeric(
      scale(score_tactile_disalienation + 0.5 * score_bodily_autonomy) -
      scale(score_carceral_containment + 0.5 * score_state_bureaucracy)
    )
  )

# ------------------------------------------------------------------------------
# 4. UMAP MANIFOLD PROJECTION & K-MEANS VISUAL CLUSTERING (SECTION 5.2)
# ------------------------------------------------------------------------------
cat("[INFO] Extracting 768-D CLIP matrix for UMAP manifold projection...\n")
clip_mat <- raw_clip_df %>%
  select(starts_with("clip_")) %>%
  as.matrix()

stopifnot(ncol(clip_mat) == 768)

set.seed(42)
n_rows <- nrow(clip_mat)
n_neighbors_val <- min(30L, max(5L, floor(n_rows / 4)))

cat(sprintf("[INFO] Running uwot::umap on %d x 768 matrix (n_neighbors=%d, metric='cosine')...\n",
            n_rows, n_neighbors_val))
umap_coords <- uwot::umap(
  clip_mat,
  n_neighbors = n_neighbors_val,
  n_components = 2,
  min_dist = 0.12,
  metric = "cosine",
  n_threads = parallel::detectCores(),
  verbose = FALSE
)

# Assign K = 6 Macro-Visual Clusters on the CLIP representation
k_clusters <- min(6L, max(2L, floor(n_rows / 10)))
set.seed(42)
km_fit <- kmeans(clip_mat, centers = k_clusters, nstart = 20, iter.max = 100)

tidy_corpus <- metadata_df %>%
  mutate(
    umap_1 = umap_coords[, 1],
    umap_2 = umap_coords[, 2],
    visual_cluster_id = km_fit$cluster
  )

cluster_profiles <- tidy_corpus %>%
  group_by(visual_cluster_id) %>%
  summarise(
    n_artworks = n(),
    share_pct = round(100 * n() / nrow(tidy_corpus), 2),
    mean_surveillance_biometric = mean(score_surveillance_biometric, na.rm = TRUE),
    mean_carceral_containment   = mean(score_carceral_containment, na.rm = TRUE),
    mean_bodily_autonomy        = mean(score_bodily_autonomy, na.rm = TRUE),
    mean_tactile_disalienation  = mean(score_tactile_disalienation, na.rm = TRUE),
    mean_digital_mediation      = mean(score_digital_mediation, na.rm = TRUE),
    mean_state_bureaucracy      = mean(score_state_bureaucracy, na.rm = TRUE),
    mean_disalienation_index    = mean(disalienation_index, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(
    visual_cluster = sprintf("Cluster %d", visual_cluster_id)
  )

tidy_corpus <- tidy_corpus %>%
  left_join(
    cluster_profiles %>% select(visual_cluster_id, visual_cluster),
    by = "visual_cluster_id"
  )

saveRDS(tidy_corpus, "data/pcap_tidy_corpus.rds")
readr::write_csv(cluster_profiles, "results/charts/table02_cluster_prompt_profiles.csv")
cat("[TDD PASS] Saved data/pcap_tidy_corpus.rds and results/charts/table02_cluster_prompt_profiles.csv\n")

# ------------------------------------------------------------------------------
# 5. SECTION 5.1 TAXONOMY TABLE & HYPOTHESIS TEST (TABLE 01 & FIGURE 01)
# ------------------------------------------------------------------------------
kw_test <- kruskal.test(score_tactile_disalienation ~ medium_category, data = tidy_corpus)
cat(sprintf("[INFO] Section 5.1 Kruskal-Wallis Test across Mediums: H = %.3f, p = %.4e\n",
            kw_test$statistic, kw_test$p.value))

table01_medium_taxonomy <- tidy_corpus %>%
  group_by(medium_category) %>%
  summarise(
    n_artworks = n(),
    corpus_percentage = round(100 * n() / nrow(tidy_corpus), 2),
    mean_tactile_disalienation = round(mean(score_tactile_disalienation, na.rm = TRUE), 4),
    sd_tactile_disalienation   = round(sd(score_tactile_disalienation, na.rm = TRUE), 4),
    mean_bodily_autonomy       = round(mean(score_bodily_autonomy, na.rm = TRUE), 4),
    mean_surveillance_score    = round(mean(score_surveillance_biometric, na.rm = TRUE), 4),
    mean_bureaucracy_score     = round(mean(score_state_bureaucracy, na.rm = TRUE), 4),
    mean_disalienation_index   = round(mean(disalienation_index, na.rm = TRUE), 4),
    kruskal_wallis_p           = signif(kw_test$p.value, 4),
    .groups = "drop"
  )

readr::write_csv(table01_medium_taxonomy, "results/charts/table01_pcap_medium_taxonomy.csv")
cat("[TDD PASS] Saved results/charts/table01_pcap_medium_taxonomy.csv\n")

# Figure 01: Disalienation & Tactile Vitality by Carceral Material Medium
fig01 <- ggplot(
  tidy_corpus,
  aes(x = fct_rev(medium_category), y = score_tactile_disalienation, fill = medium_category)
) +
  geom_violin(alpha = 0.45, color = NA, trim = FALSE) +
  geom_boxplot(width = 0.22, outlier.shape = NA, alpha = 0.85, color = "grey15") +
  stat_summary(fun = mean, geom = "point", shape = 23, size = 3, fill = "white") +
  coord_flip() +
  scale_fill_manual(
    values = c(
      "Contraband/Found Carceral Material" = "#B22222",
      "Mixed/Sculptural"                   = "#D95F02",
      "Graphite/Ink on Paper"              = "#7570B3",
      "Standard Paint/Canvas"              = "#1B9E77"
    ),
    guide = "none"
  ) +
  labs(
    title = "Aesthetics of Disalienation Across Carceral Material Substrates",
    subtitle = sprintf(
      "CLIP Zero-Shot Tactile Vitality / Disalienation Cosine Similarity (N = %s; Kruskal-Wallis p = %.2e)",
      scales::comma(nrow(tidy_corpus)), kw_test$p.value
    ),
    x = "Material Medium Taxonomy (Section 5.1)",
    y = "CLIP Cosine Similarity: Tactile Disalienation & Sensory Vitality",
    caption = "Source: University of Michigan PCAP Archive (ViT-L/14 Distant Viewing). White diamonds indicate group means."
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    plot.subtitle = element_text(color = "grey30", size = 10.5),
    axis.text.y = element_text(face = "bold", size = 10.5),
    panel.grid.minor = element_blank()
  )

ggsave(
  filename = "results/figures/fig01_disalienation_by_medium.png",
  plot = fig01,
  width = 10,
  height = 6,
  dpi = 300,
  bg = "white"
)

# Figure 02: UMAP Manifold Projection of the PCAP Visual Corpus
fig02 <- ggplot(
  tidy_corpus,
  aes(x = umap_1, y = umap_2, color = visual_cluster)
) +
  geom_point(alpha = 0.55, size = 1.1) +
  scale_color_brewer(palette = "Dark2", name = "Visual Cluster") +
  labs(
    title = "Distant Viewing the PCAP Corpus: 768-D CLIP UMAP Manifold",
    subtitle = sprintf(
      "2D UMAP Projection of ViT-L/14 Embeddings Across %s Incarcerated Artworks",
      scales::comma(nrow(tidy_corpus))
    ),
    x = "UMAP Dimension 1 (Cosine Metric)",
    y = "UMAP Dimension 2 (Cosine Metric)",
    caption = "Source: University of Michigan PCAP Visual Art Database (Arnold & Tilton Distant Viewing Framework)."
  ) +
  guides(color = guide_legend(override.aes = list(size = 3.5, alpha = 0.95))) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    plot.subtitle = element_text(color = "grey30", size = 10.5),
    legend.position = "right",
    panel.grid.minor = element_blank()
  )

ggsave(
  filename = "results/figures/fig02_umap_visual_clusters.png",
  plot = fig02,
  width = 10,
  height = 7,
  dpi = 300,
  bg = "white"
)

# Figure 02b: Small-Multiple Faceted UMAP by Material Medium
# Replicates Arnold & Tilton (2023, Chapter 6, Figure 6.6, p. 210)
umap_bg_df <- tidy_corpus %>% select(umap_1, umap_2)

fig02b <- ggplot(tidy_corpus, aes(x = umap_1, y = umap_2)) +
  geom_point(
    data = umap_bg_df,
    aes(x = umap_1, y = umap_2),
    color = "grey85",
    size = 0.5,
    alpha = 0.45
  ) +
  geom_point(aes(color = medium_category), size = 0.9, alpha = 0.80) +
  facet_wrap(~ medium_category, ncol = 2) +
  scale_color_manual(
    values = c(
      "Contraband/Found Carceral Material" = "#B22222",
      "Mixed/Sculptural"                   = "#D95F02",
      "Graphite/Ink on Paper"              = "#7570B3",
      "Standard Paint/Canvas"              = "#1B9E77"
    ),
    guide = "none"
  ) +
  labs(
    title = "Material Substrates Across the CLIP Visual Embedding Manifold",
    subtitle = "Each panel highlights one material medium category against the full PCAP corpus in light gray (Arnold & Tilton, 2023, p. 210)",
    x = "UMAP Dimension 1 (Cosine Metric)",
    y = "UMAP Dimension 2 (Cosine Metric)"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    strip.text = element_text(face = "bold", size = 11),
    panel.grid.minor = element_blank()
  )

ggsave(
  filename = "results/figures/fig02b_umap_faceted_by_medium.png",
  plot = fig02b,
  width = 10,
  height = 8,
  dpi = 300,
  bg = "white"
)

stopifnot(
  file.exists("data/pcap_tidy_corpus.rds"),
  file.exists("results/charts/table01_pcap_medium_taxonomy.csv"),
  file.exists("results/charts/table02_cluster_prompt_profiles.csv"),
  file.exists("results/figures/fig01_disalienation_by_medium.png"),
  file.exists("results/figures/fig02_umap_visual_clusters.png")
)
cat("[TDD PASS] Script 04 Complete! All 5 target artifacts verified.\n")

[INFO] SCRIPT 04: Tidy Metadata Normalization, Disalienation Test & CLIP UMAP
[INFO] Ingested 13546 artworks and 789 columns from data/pcap_clip_embeddings.parquet.
[TDD PASS] Metadata normalization helper functions verified.
[INFO] Extracting 768-D CLIP matrix for UMAP manifold projection...
[INFO] Running uwot::umap on 13546 x 768 matrix (n_neighbors=30, metric='cosine')...
[TDD PASS] Saved data/pcap_tidy_corpus.rds and results/charts/table02_cluster_prompt_profiles.csv
[INFO] Section 5.1 Kruskal-Wallis Test across Mediums: H = 360.840, p = 6.7032e-78
[TDD PASS] Saved results/charts/table01_pcap_medium_taxonomy.csv
[TDD PASS] Script 04 Complete! All 5 target artifacts verified.


---
## Step 05: LLaVA Tidy NLP, Visual Plural Bypass & STM `searchK` (`05_llava_tidy_nlp_and_searchK.R`)
Fault-tolerantly ingests `data/pcap_llava_interrogations.jsonl`, joins with `data/pcap_tidy_corpus.rds`, performs `tidytext` tokenization and `textstem` lemmatization with pre- and post-lemmatization "Visual Plural Bypass" stop-word pruning, builds the STM document-term matrix, and runs `stm::searchK` across $K \in \{10, 15, 20, 25, 30, 40\}$.

**Outputs:** `data/pcap_llava_stm_out.rds`, `data/pcap_llava_searchK.rds`, `results/figures/fig03_llava_stm_searchK.png`

In [ ]:
%%R
# ==============================================================================
# scripts/05_llava_tidy_nlp_and_searchK.R
# Cultural Resistance: Counter-Visuality in the PCAP Archive
# PURPOSE: Fault-tolerantly ingests data/pcap_llava_interrogations.jsonl, joins
#          with data/pcap_tidy_corpus.rds, executes tidytext tokenization and
#          textstem lemmatization, and applies the "Visual Plural Bypass"
#          stop-word filter both BEFORE and AFTER lemmatization. Builds the stm
#          document-term object (out_visual) and runs searchK across
#          K in {10, 15, 20, 25, 30, 40}.
# OUTPUTS: data/pcap_llava_stm_out.rds
#          data/pcap_llava_searchK.rds
#          results/figures/fig03_llava_stm_searchK.png
# ==============================================================================

suppressPackageStartupMessages({
  library(tidyverse)
  library(jsonlite)
  library(tidytext)
  library(textstem)
  library(stm)
  library(patchwork)
})

cat("==============================================================================\n")
cat("[INFO] SCRIPT 05: LLaVA Tidy NLP, Visual Plural Bypass & STM searchK\n")
cat("==============================================================================\n")

if (dir.exists("/content/drive/MyDrive/colab_pcap_folder")) {
  setwd("/content/drive/MyDrive/colab_pcap_folder")
}

jsonl_path <- "data/pcap_llava_interrogations.jsonl"
corpus_rds_path <- "data/pcap_tidy_corpus.rds"

if (!file.exists(jsonl_path) || !file.exists(corpus_rds_path)) {
  stop("[ERROR] Missing data/pcap_llava_interrogations.jsonl or data/pcap_tidy_corpus.rds!")
}

# ------------------------------------------------------------------------------
# 1. FAULT-TOLERANT LINE-BY-LINE JSONL INGESTION
# ------------------------------------------------------------------------------
raw_lines <- readLines(jsonl_path, warn = FALSE, encoding = "UTF-8")
parsed_records <- list()

for (i in seq_along(raw_lines)) {
  ln <- str_trim(raw_lines[[i]])
  if (nchar(ln) == 0) next
  rec <- tryCatch(jsonlite::fromJSON(ln), error = function(e) NULL)
  if (!is.null(rec) && !is.null(rec$item_id) && !is.null(rec$llava_description)) {
    parsed_records[[length(parsed_records) + 1]] <- as_tibble(rec)
  }
}

llava_df <- bind_rows(parsed_records) %>%
  mutate(item_id = as.character(item_id)) %>%
  distinct(item_id, .keep_all = TRUE) %>%
  filter(nchar(str_trim(llava_description)) > 20)

cat(sprintf("[INFO] Fault-tolerantly ingested %d unique LLaVA transcripts.\n", nrow(llava_df)))

tidy_corpus <- readRDS(corpus_rds_path) %>%
  mutate(item_id = as.character(item_id))

joined_df <- llava_df %>%
  select(
    item_id,
    sampling_stratum,
    llava_description,
    any_of(c("formal_brightness", "formal_saturation", "formal_warm_ratio"))
  ) %>%
  inner_join(tidy_corpus, by = "item_id")

stopifnot(nrow(joined_df) > 0)
cat(sprintf("[INFO] Joined LLaVA transcripts with tidy metadata: N = %d artworks.\n", nrow(joined_df)))

# ------------------------------------------------------------------------------
# 2. THE "VISUAL PLURAL BYPASS" STOP-WORD FILTER (PRE- AND POST-LEMMATIZATION)
# ------------------------------------------------------------------------------
visual_vlm_boilerplate <- unique(c(
  "image", "images", "painting", "paintings", "drawing", "drawings",
  "artwork", "artworks", "depict", "depicts", "depicted", "depicting",
  "depiction", "depictions", "show", "shows", "shown", "showing",
  "feature", "features", "featured", "featuring", "background", "backgrounds",
  "foreground", "foregrounds", "color", "colors", "colored", "colour", "colours",
  "style", "styles", "piece", "pieces", "artist", "artists", "viewer", "viewers",
  "composition", "compositions", "overall", "appear", "appears", "appeared",
  "appearing", "visible", "create", "creates", "created", "creating",
  "sense", "senses", "convey", "conveys", "conveyed", "conveying",
  "suggest", "suggests", "suggested", "suggesting", "scene", "scenes",
  "element", "elements", "detail", "details", "detailed", "include", "includes",
  "included", "including", "seen", "see", "visual", "visually", "form", "forms",
  "formal", "part", "parts", "section", "sections", "aspect", "aspects",
  "subject", "subjects", "medium", "media", "work", "works", "art", "picture",
  "pictures", "illustration", "illustrations", "portrait", "portraits",
  "first", "second", "third", "fourth", "one", "two", "three", "four",
  "left", "right", "center", "central", "top", "bottom", "side", "sides",
  "middle", "area", "areas", "space", "spaces", "use", "uses", "used", "using",
  # Pilot-Identified Spatial & COCO Domain-Shift Boilerplate
  "locate", "located", "location", "setting", "settings", "environment",
  "environments", "corner", "corners", "frame", "frames", "framed",
  "individual", "individuals", "addition", "additionally", "main", "possibly",
  "possible", "further", "closer", "towards", "situate", "situated", "place",
  "placed", "surround", "surrounded", "surrounding", "relax", "relaxation",
  "socialize", "socializing", "public", "gather", "gathering", "represent",
  "represents", "represented", "representing", "paragraph"
))

data("stop_words", package = "tidytext")
master_stop_vocab <- unique(c(stop_words$word, visual_vlm_boilerplate))

# TDD Self-Test: Verify Visual Plural Bypass eliminates both "paintings" and "depicting"
test_tokens <- tibble(word = c("paintings", "depicting", "watchtowers", "barcodes")) %>%
  filter(!word %in% master_stop_vocab) %>%
  mutate(lemma = textstem::lemmatize_words(word)) %>%
  filter(!lemma %in% master_stop_vocab)

stopifnot(
  !("painting" %in% test_tokens$lemma),
  !("depict" %in% test_tokens$lemma),
  all(c("watchtower", "barcode") %in% test_tokens$lemma)
)
cat("[TDD PASS] 'Visual Plural Bypass' pre- and post-lemmatization stop-word gate verified.\n")

# Apply tokenization -> Pre-Filter -> Lemmatization -> Post-Filter
cleaned_tokens <- joined_df %>%
  select(item_id, llava_description) %>%
  unnest_tokens(output = word, input = llava_description, token = "words") %>%
  filter(str_detect(word, "^[a-z]{3,}$")) %>%
  # Gate 1: Pre-lemmatization filter (catches plural/inflected VLM boilerplate)
  filter(!word %in% master_stop_vocab) %>%
  mutate(lemma = textstem::lemmatize_words(word)) %>%
  # Gate 2: Post-lemmatization filter (catches normalized stems)
  filter(!lemma %in% master_stop_vocab) %>%
  filter(nchar(lemma) >= 3)

reconstructed_docs <- cleaned_tokens %>%
  group_by(item_id) %>%
  summarise(cleaned_text = paste(lemma, collapse = " "), .groups = "drop")

stm_input_df <- joined_df %>%
  inner_join(reconstructed_docs, by = "item_id") %>%
  filter(nchar(cleaned_text) > 10)

# ------------------------------------------------------------------------------
# 3. BUILD STM CORPUS OBJECT (out_visual)
# ------------------------------------------------------------------------------
cat("[INFO] Building STM document-term matrix via textProcessor & prepDocuments...\n")
processed <- stm::textProcessor(
  documents = stm_input_df$cleaned_text,
  metadata = stm_input_df,
  lowercase = FALSE,
  removestopwords = FALSE,
  removenumbers = TRUE,
  removepunctuation = FALSE,
  stem = FALSE,
  verbose = FALSE
)

lower_thresh_val <- ifelse(length(processed$documents) >= 200, 3L, 1L)
out_visual <- stm::prepDocuments(
  documents = processed$documents,
  vocab = processed$vocab,
  meta = processed$meta,
  lower.thresh = lower_thresh_val,
  verbose = FALSE
)

saveRDS(out_visual, "data/pcap_llava_stm_out.rds")
cat(sprintf(
  "[TDD PASS] Saved data/pcap_llava_stm_out.rds (%d documents, %d vocabulary terms).\n",
  length(out_visual$documents), length(out_visual$vocab)
))

# ------------------------------------------------------------------------------
# 4. RUN STM searchK DIAGNOSTICS ACROSS K IN {10, 15, 20, 25, 30, 40}
# ------------------------------------------------------------------------------
candidate_K <- c(10, 15, 20, 25, 30, 40)
cat(sprintf("[INFO] Running stm::searchK across K = {%s}...\n",
            paste(candidate_K, collapse = ", ")))

# Adapt B-spline degrees of freedom safely if running on a small pilot subset
n_unique_yrs <- length(unique(out_visual$meta$clean_year_stm))
prevalence_formula <- if (n_unique_yrs >= 10) {
  ~ s(clean_year_stm) + medium_category
} else {
  ~ clean_year_stm + medium_category
}

set.seed(42)
searchk_fit <- stm::searchK(
  documents = out_visual$documents,
  vocab = out_visual$vocab,
  K = candidate_K,
  prevalence = prevalence_formula,
  data = out_visual$meta,
  init.type = "Spectral",
  verbose = FALSE
)

saveRDS(searchk_fit, "data/pcap_llava_searchK.rds")

searchk_metrics <- tibble(
  K        = unlist(searchk_fit$results$K),
  heldout  = unlist(searchk_fit$results$heldout),
  residual = unlist(searchk_fit$results$residual),
  semcoh   = unlist(searchk_fit$results$semcoh),
  exclus   = unlist(searchk_fit$results$exclus)
)

# ------------------------------------------------------------------------------
# 5. PLOT 4-PANEL searchK DIAGNOSTIC FIGURE (FIGURE 03)
# ------------------------------------------------------------------------------
make_diag_panel <- function(df, y_col, title_str, y_lab, color_hex) {
  ggplot(df, aes(x = K, y = .data[[y_col]])) +
    geom_vline(xintercept = 25, linetype = "dashed", color = "#B22222", linewidth = 0.8) +
    geom_line(color = color_hex, linewidth = 1.1) +
    geom_point(size = 3, color = color_hex) +
    scale_x_continuous(breaks = candidate_K) +
    labs(title = title_str, x = "Number of Topics (K)", y = y_lab) +
    theme_minimal(base_size = 11) +
    theme(plot.title = element_text(face = "bold", size = 11.5))
}

p1 <- make_diag_panel(searchk_metrics, "heldout",  "A. Held-Out Likelihood", "Log-Likelihood", "#1B9E77")
p2 <- make_diag_panel(searchk_metrics, "exclus",   "B. Topic Exclusivity",   "FREX Exclusivity", "#D95F02")
p3 <- make_diag_panel(searchk_metrics, "semcoh",   "C. Semantic Coherence",  "Coherence Score", "#7570B3")
p4 <- make_diag_panel(searchk_metrics, "residual", "D. Dispersion Residuals", "Residual Variance", "#E7298A")

fig03 <- (p1 + p2) / (p3 + p4) +
  plot_annotation(
    title = "Structural Topic Model (STM) Hyperparameter Validation on LLaVA Visual Descriptions",
    subtitle = "Spectral Initialization across K in {10, 15, 20, 25, 30, 40}; Red dashed line indicates target K = 25",
    caption = "Source: PCAP Stratified High-Signal LLaVA Subsample (Visual Plural Bypass Filtered)."
  )

ggsave(
  filename = "results/figures/fig03_llava_stm_searchK.png",
  plot = fig03,
  width = 10,
  height = 7,
  dpi = 300,
  bg = "white"
)

stopifnot(
  file.exists("data/pcap_llava_stm_out.rds"),
  file.exists("data/pcap_llava_searchK.rds"),
  file.exists("results/figures/fig03_llava_stm_searchK.png")
)
cat("[TDD PASS] Script 05 Complete! All 3 target artifacts verified.\n")

[INFO] SCRIPT 05: LLaVA Tidy NLP, Visual Plural Bypass & STM searchK
[INFO] Fault-tolerantly ingested 2174 unique LLaVA transcripts.
[INFO] Joined LLaVA transcripts with tidy metadata: N = 2174 artworks.
[TDD PASS] 'Visual Plural Bypass' pre- and post-lemmatization stop-word gate verified.
[INFO] Building STM document-term matrix via textProcessor & prepDocuments...
[TDD PASS] Saved data/pcap_llava_stm_out.rds (2174 documents, 1181 vocabulary terms).
[INFO] Running stm::searchK across K = {10, 15, 20, 25, 30, 40}...
[TDD PASS] Script 05 Complete! All 3 target artifacts verified.


---
## Step 06: Deterministic $K = 25$ Spectral STM on Visual Motifs (`06_stm_visual_motifs_k25.R`)
Trains the deterministic $K = 25$ Structural Topic Model (`init.type = "Spectral"`) on the cleaned LLaVA iconographic descriptions (`prevalence = ~ s(clean_year) + medium_category`), extracts `FREX`, `Highest Prob`, `Lift`, and `Score` word matrices (`stm::labelTopics`), and maps the core counter-visual motif families (Section 5.3).

**Outputs:** `data/pcap_stm_visual_k25.rds`, `results/charts/table03_llava_visual_topics.csv`

In [ ]:
%%R
# ==============================================================================
# scripts/06_stm_visual_motifs_k25.R
# Cultural Resistance: Counter-Visuality in the PCAP Archive
# PURPOSE: Trains the deterministic K = 25 Structural Topic Model
#          (init.type = "Spectral") on the cleaned LLaVA iconographic
#          descriptions with prevalence = ~ s(clean_year) + medium_category.
#          Extracts FREX, Highest Prob, Lift, and Score word matrices
#          (labelTopics) and maps the core counter-visual motifs (Section 5.3).
# OUTPUTS: data/pcap_stm_visual_k25.rds
#          results/charts/table03_llava_visual_topics.csv
# ==============================================================================

suppressPackageStartupMessages({
  library(tidyverse)
  library(stm)
})

cat("==============================================================================\n")
cat("[INFO] SCRIPT 06: Deterministic K = 25 Spectral STM on LLaVA Visual Motifs\n")
cat("==============================================================================\n")

if (dir.exists("/content/drive/MyDrive/colab_pcap_folder")) {
  setwd("/content/drive/MyDrive/colab_pcap_folder")
}

stm_out_path <- "data/pcap_llava_stm_out.rds"
if (!file.exists(stm_out_path)) {
  stop("[ERROR] Missing data/pcap_llava_stm_out.rds. Run Script 05 first!")
}

out_visual <- readRDS(stm_out_path)
K_TARGET <- 25L

# ------------------------------------------------------------------------------
# 1. FIT DETERMINISTIC SPECTRAL STM (K = 25)
# ------------------------------------------------------------------------------
cat(sprintf("[INFO] Fitting Spectral STM with K = %d on %d artworks...\n",
            K_TARGET, length(out_visual$documents)))

n_unique_yrs <- length(unique(out_visual$meta$clean_year_stm))
prevalence_formula <- if (n_unique_yrs >= 10) {
  ~ s(clean_year_stm) + medium_category
} else {
  ~ clean_year_stm + medium_category
}

set.seed(42)
stm_k25 <- stm::stm(
  documents  = out_visual$documents,
  vocab      = out_visual$vocab,
  K          = K_TARGET,
  prevalence = prevalence_formula,
  data       = out_visual$meta,
  init.type  = "Spectral",
  max.em.its = 150,
  verbose    = FALSE
)

# ------------------------------------------------------------------------------
# 2. EXTRACT FREX, HIGHEST PROB, LIFT, AND SCORE MATRICES
# ------------------------------------------------------------------------------
topic_labels <- stm::labelTopics(stm_k25, n = 10)
theta_means <- colMeans(stm_k25$theta)

collapse_words <- function(mat, row_idx) {
  paste(mat[row_idx, ], collapse = ", ")
}

# Theoretical Anchor Lexicons aligned with LLaVA Concrete "Objects + Stuff"
# (Arnold & Tilton, 2023, Chapter 4, Tables 4.3-4.5, pp. 128-137)
motif_anchors <- list(
  "Panoptic Eyes & Watchtowers" = c(
    "eye", "watchtower", "tower", "camera", "gaze", "stare", "spotlight",
    "surveillance", "lens", "watch", "guard", "officer", "pupil", "iris",
    "clock", "record", "recording", "observe", "observation", "searchlight",
    "binocular", "uniform", "flashlight", "security", "look"
  ),
  "Forensic Numbers & Barcodes" = c(
    "number", "numeral", "barcode", "digit", "stamp", "code", "label",
    "identification", "serial", "form", "text", "letter", "grid", "placard",
    "tally", "sign", "word", "written", "writing", "document", "badge",
    "calendar", "mark", "inscription", "tag"
  ),
  "Fragmented/Shackled Bodies & Counter-Gaze" = c(
    "chain", "shackle", "handcuff", "bar", "wire", "razor", "cage", "cell",
    "face", "hand", "body", "expression", "defiant", "direct", "tear", "scream",
    "brick", "wall", "rope", "pole", "suspend", "hang", "hanging", "neck",
    "waist", "wrist", "bound", "tie", "fence", "lock", "door", "blood", "fist"
  ),
  "Pastoral Escape & Tactile Disalienation" = c(
    "tree", "sky", "horizon", "mountain", "water", "bird", "flower", "nature",
    "forest", "sun", "sunset", "landscape", "cloud", "river", "texture",
    "impasto", "carve", "waterfall", "stream", "pine", "grass", "field",
    "ocean", "lake", "leaf", "branch", "butterfly", "horse", "wolf", "animal",
    "yarn", "fabric", "collage", "woven", "soap", "wood", "thread"
  ),
  "Telephonic/Screen Separation & Digital Mediation" = c(
    "phone", "telephone", "handset", "screen", "monitor", "television",
    "glass", "plexiglass", "window", "booth", "cord", "cable", "digital",
    "pixel", "electronic", "computer", "visitation", "microphone", "speaker",
    "keyboard", "button", "device", "machine", "divider", "partition"
  )
)

score_motif_match <- function(words_vec, anchor_vec) {
  sum(words_vec %in% anchor_vec)
}

topic_rows <- vector("list", K_TARGET)
for (k in seq_len(K_TARGET)) {
  frex_vec  <- topic_labels$frex[k, ]
  prob_vec  <- topic_labels$prob[k, ]
  combined_top <- unique(c(frex_vec, prob_vec))

  overlap_scores <- vapply(
    motif_anchors,
    function(anchors) score_motif_match(combined_top, anchors),
    numeric(1)
  )
  best_family <- names(which.max(overlap_scores))
  if (max(overlap_scores) == 0) {
    best_family <- "General Carceral Iconography"
  }

  topic_rows[[k]] <- tibble(
    topic_id           = k,
    topic_label        = sprintf("Topic %02d: %s (%s)", k, best_family, paste(frex_vec[1:3], collapse = "/")),
    theoretical_family = best_family,
    anchor_overlap     = max(overlap_scores),
    expected_prop      = round(theta_means[k], 5),
    expected_pct       = round(100 * theta_means[k], 2),
    frex_top10         = collapse_words(topic_labels$frex, k),
    prob_top10         = collapse_words(topic_labels$prob, k),
    lift_top10         = collapse_words(topic_labels$lift, k),
    score_top10        = collapse_words(topic_labels$score, k)
  )
}

table03_topics <- bind_rows(topic_rows) %>%
  arrange(desc(expected_prop))

# Attach topic table metadata inside the saved RDS list for downstream scripts (07-09)
attr(stm_k25, "topic_summary_table") <- table03_topics
saveRDS(stm_k25, "data/pcap_stm_visual_k25.rds")
readr::write_csv(table03_topics, "results/charts/table03_llava_visual_topics.csv")

stopifnot(
  file.exists("data/pcap_stm_visual_k25.rds"),
  file.exists("results/charts/table03_llava_visual_topics.csv"),
  nrow(table03_topics) == 25
)
cat("[TDD PASS] Script 06 Complete! Saved data/pcap_stm_visual_k25.rds & table03_llava_visual_topics.csv.\n")

[INFO] SCRIPT 06: Deterministic K = 25 Spectral STM on LLaVA Visual Motifs
[INFO] Fitting Spectral STM with K = 25 on 2174 artworks...
[TDD PASS] Script 06 Complete! Saved data/pcap_stm_visual_k25.rds & table03_llava_visual_topics.csv.


---
## Step 07: Counter-Visuality B-Splines & Surge-Era Regressions (`07_countervisuality_regressions.R`)
Estimates B-spline temporal trajectories (`stm::estimateEffect`) and OLS linear models (`lm`, `broom`, `modelsummary`, `gt`) across both the full archive timeline (1996–2024) and the Carceral-Digital Turn surge era (2010–2024) (Section 5.4: Mirzoeff & Browne).

**Outputs:** `results/figures/fig04_countervisuality_trajectories.png`, `results/charts/table04_temporal_regressions.csv`, `results/charts/table04_temporal_regressions.html`

In [ ]:
%%R
# ==============================================================================
# scripts/07_countervisuality_regressions.R
# Cultural Resistance: Counter-Visuality in the PCAP Archive
# PURPOSE: Estimates B-spline temporal trajectories (stm::estimateEffect) and
#          OLS linear models (lm + broom + modelsummary + gt) across both the
#          full archive timeline (1996-2024) and the Carceral-Digital Turn
#          surge era (2010-2024). Tests whether biometric/optical surveillance
#          and digital mediation motifs surge alongside prison digitization
#          (Section 5.4: Mirzoeff & Browne).
# OUTPUTS: results/figures/fig04_countervisuality_trajectories.png
#          results/charts/table04_temporal_regressions.csv
#          results/charts/table04_temporal_regressions.html
# ==============================================================================

suppressPackageStartupMessages({
  library(tidyverse)
  library(stm)
  library(broom)
  library(modelsummary)
  library(gt)
  library(splines)
  library(scales)
})

cat("==============================================================================\n")
cat("[INFO] SCRIPT 07: Temporal B-Splines & Surge-Era Counter-Visuality Regressions\n")
cat("==============================================================================\n")

if (dir.exists("/content/drive/MyDrive/colab_pcap_folder")) {
  setwd("/content/drive/MyDrive/colab_pcap_folder")
}

stm_k25 <- readRDS("data/pcap_stm_visual_k25.rds")
out_visual <- readRDS("data/pcap_llava_stm_out.rds")
tidy_corpus <- readRDS("data/pcap_tidy_corpus.rds")
table03_topics <- attr(stm_k25, "topic_summary_table")

# ------------------------------------------------------------------------------
# 1. ESTIMATE B-SPLINE TEMPORAL EFFECTS (stm::estimateEffect)
# ------------------------------------------------------------------------------
cat("[INFO] Running stm::estimateEffect across all 25 visual motifs...\n")
n_unique_yrs <- length(unique(out_visual$meta$clean_year_stm))
effect_formula <- if (n_unique_yrs >= 10) {
  1:25 ~ s(clean_year_stm) + medium_category
} else {
  1:25 ~ clean_year_stm + medium_category
}

set.seed(42)
prep_effects <- stm::estimateEffect(
  formula     = effect_formula,
  stmobj      = stm_k25,
  metadata    = out_visual$meta,
  uncertainty = "Global"
)

# ------------------------------------------------------------------------------
# 2. CONSTRUCT DOCUMENT-LEVEL COUNTER-VISUALITY INDICES (VERIFIED YEARS ONLY)
# ------------------------------------------------------------------------------
theta_mat <- as.data.frame(stm_k25$theta)
colnames(theta_mat) <- sprintf("topic_%02d", 1:25)

get_family_topics <- function(fam_name, fallback_idx) {
  matched <- table03_topics %>%
    filter(theoretical_family == fam_name) %>%
    pull(topic_id)
  if (length(matched) == 0) return(fallback_idx)
  return(matched)
}

idx_panoptic   <- get_family_topics("Panoptic Eyes & Watchtowers", c(1, 2))
idx_forensic   <- get_family_topics("Forensic Numbers & Barcodes", c(3, 4))
idx_shackled   <- get_family_topics("Fragmented/Shackled Bodies & Counter-Gaze", c(5, 6))
idx_telephonic <- get_family_topics("Telephonic/Screen Separation & Digital Mediation", c(7, 8))

# Filter strictly to verified (non-imputed) exhibition years (!year_imputed)
reg_df <- out_visual$meta %>%
  as_tibble() %>%
  bind_cols(theta_mat) %>%
  filter(!year_imputed, !is.na(clean_year), clean_year >= 1996, clean_year <= 2025) %>%
  mutate(
    clean_year = as.numeric(clean_year),
    year_centered_1996 = clean_year - 1996,
    post_2010_digital_turn = ifelse(clean_year >= 2010, 1L, 0L),
    contraband_found_medium = ifelse(
      medium_category == "Contraband/Found Carceral Material", 1L, 0L
    ),
    stm_panoptic_surveillance  = rowSums(across(all_of(sprintf("topic_%02d", idx_panoptic)))),
    stm_forensic_biometric     = rowSums(across(all_of(sprintf("topic_%02d", idx_forensic)))),
    stm_shackled_countergaze   = rowSums(across(all_of(sprintf("topic_%02d", idx_shackled)))),
    stm_digital_telephonic     = rowSums(across(all_of(sprintf("topic_%02d", idx_telephonic)))),
    stm_total_countervisuality = stm_panoptic_surveillance + stm_forensic_biometric + stm_digital_telephonic
  )

# Half-Decade Summary Table (Arnold & Tilton, 2023, Table 3.1, p. 71)
table04b_half_decade <- reg_df %>%
  group_by(half_decade) %>%
  summarise(
    n_artworks             = n(),
    mean_panoptic_pct      = round(100 * mean(stm_panoptic_surveillance, na.rm = TRUE), 2),
    mean_forensic_pct      = round(100 * mean(stm_forensic_biometric, na.rm = TRUE), 2),
    mean_shackled_pct      = round(100 * mean(stm_shackled_countergaze, na.rm = TRUE), 2),
    mean_digital_pct       = round(100 * mean(stm_digital_telephonic, na.rm = TRUE), 2),
    mean_total_counter_pct = round(100 * mean(stm_total_countervisuality, na.rm = TRUE), 2),
    .groups = "drop"
  )
readr::write_csv(table04b_half_decade, "results/charts/table04b_half_decade_trends.csv")

# ------------------------------------------------------------------------------
# 3. FIT FULL-ARCHIVE (1996-2024) AND SURGE-ERA (2010-2024) OLS REGRESSIONS
# ------------------------------------------------------------------------------
m1_full_stm <- lm(
  stm_total_countervisuality ~ year_centered_1996 + contraband_found_medium + score_bodily_autonomy,
  data = reg_df
)

reg_df_surge <- reg_df %>% filter(clean_year >= 2010)
if (nrow(reg_df_surge) < 10) reg_df_surge <- reg_df

m2_surge_stm <- lm(
  stm_total_countervisuality ~ I(clean_year - 2010) + contraband_found_medium + score_bodily_autonomy,
  data = reg_df_surge
)

tidy_full_reg <- tidy_corpus %>%
  filter(!year_imputed, !is.na(clean_year), clean_year >= 1996, clean_year <= 2025) %>%
  mutate(
    year_centered_1996 = clean_year - 1996,
    contraband_found_medium = ifelse(
      medium_category == "Contraband/Found Carceral Material", 1L, 0L
    )
  )

m3_full_clip_surv <- lm(
  score_surveillance_biometric ~ year_centered_1996 + contraband_found_medium + score_bodily_autonomy,
  data = tidy_full_reg
)

tidy_surge_reg <- tidy_full_reg %>% filter(clean_year >= 2010)
if (nrow(tidy_surge_reg) < 10) tidy_surge_reg <- tidy_full_reg

m4_surge_clip_digital <- lm(
  I(score_digital_mediation + score_state_bureaucracy) ~ I(clean_year - 2010) + contraband_found_medium + score_bodily_autonomy,
  data = tidy_surge_reg
)

models_list <- list(
  "M1: STM Counter-Visuality (1996-2024)"     = m1_full_stm,
  "M2: STM Digital Turn Surge (2010-2024)"    = m2_surge_stm,
  "M3: CLIP Panoptic Surveillance (1996-2024)" = m3_full_clip_surv,
  "M4: CLIP Digital/Biometric (2010-2024)"    = m4_surge_clip_digital
)

table04_csv <- bind_rows(
  lapply(names(models_list), function(m_name) {
    broom::tidy(models_list[[m_name]], conf.int = TRUE) %>%
      mutate(model = m_name, .before = 1)
  })
)
readr::write_csv(table04_csv, "results/charts/table04_temporal_regressions.csv")

ms_gt_table <- modelsummary::modelsummary(
  models_list,
  output = "gt",
  stars = c("*" = 0.05, "**" = 0.01, "***" = 0.001),
  gof_omit = "IC|Log|F|RMSE",
  title = "Table 04: Temporal OLS Regressions of Counter-Visuality & Biometric Subversion (1996–2024 vs. 2010–2024)"
)
gt::gtsave(ms_gt_table, filename = "results/charts/table04_temporal_regressions.html")

# ------------------------------------------------------------------------------
# 4. PLOT B-SPLINE TEMPORAL TRAJECTORIES (FIGURE 04)
# ------------------------------------------------------------------------------
trajectory_long <- reg_df %>%
  select(
    clean_year,
    `1. Panoptic Eyes & Watchtowers`       = stm_panoptic_surveillance,
    `2. Forensic Numbers & Barcodes`       = stm_forensic_biometric,
    `3. Shackled Bodies & Counter-Gaze`    = stm_shackled_countergaze,
    `4. Telephonic & Digital Mediation`    = stm_digital_telephonic
  ) %>%
  pivot_longer(
    cols = -clean_year,
    names_to = "motif_dimension",
    values_to = "topic_proportion"
  )

fig04 <- ggplot(
  trajectory_long,
  aes(x = clean_year, y = topic_proportion, color = motif_dimension, fill = motif_dimension)
) +
  geom_vline(xintercept = 2010, linetype = "dashed", color = "grey20", linewidth = 0.8) +
  annotate(
    "text",
    x = 2010.4,
    y = max(trajectory_long$topic_proportion, na.rm = TRUE) * 0.92,
    label = "Carceral-Digital Turn (2010)",
    hjust = 0,
    size = 3.3,
    fontface = "italic",
    color = "grey20"
  ) +
  geom_smooth(
    method = "lm",
    formula = y ~ splines::bs(x, df = 4),
    se = TRUE,
    alpha = 0.18,
    linewidth = 1.15
  ) +
  facet_wrap(~ motif_dimension, scales = "free_y", ncol = 2) +
  scale_color_brewer(palette = "Dark2", guide = "none") +
  scale_fill_brewer(palette = "Dark2", guide = "none") +
  scale_y_continuous(labels = scales::percent_format(accuracy = 0.1)) +
  labs(
    title = "Counter-Visual Trajectories in the PCAP Archive (1996–2024)",
    subtitle = "B-Spline Smoothed STM Motif Prevalence Across the Rollout of Carceral Digital Infrastructure",
    x = "PCAP Annual Exhibition Year",
    y = "Expected Visual Motif Proportion (STM Theta)",
    caption = "Source: University of Michigan PCAP Archive (K = 25 LLaVA STM). Shaded bands indicate 95% confidence intervals."
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    strip.text = element_text(face = "bold", size = 11),
    panel.grid.minor = element_blank()
  )

ggsave(
  filename = "results/figures/fig04_countervisuality_trajectories.png",
  plot = fig04,
  width = 11,
  height = 7.5,
  dpi = 300,
  bg = "white"
)

stopifnot(
  file.exists("results/figures/fig04_countervisuality_trajectories.png"),
  file.exists("results/charts/table04_temporal_regressions.csv"),
  file.exists("results/charts/table04_temporal_regressions.html")
)
cat("[TDD PASS] Script 07 Complete! All 3 target artifacts verified.\n")

[INFO] SCRIPT 07: Temporal B-Splines & Surge-Era Counter-Visuality Regressions
[INFO] Running stm::estimateEffect across all 25 visual motifs...
[TDD PASS] Script 07 Complete! All 3 target artifacts verified.


---
## Step 08: Cross-Modal Network Topology (`08_crossmodal_network_igraph.R`)
Computes the STM topic correlation matrix (`stm::topicCorr`) across the 25 LLaVA visual motifs and links CLIP visual clusters and carceral material medium covariates into a multipartite `igraph` network visualization.

**Outputs:** `data/pcap_visual_topic_corr.rds`, `results/figures/fig05_crossmodal_network_igraph.png`, `results/charts/table05_material_motif_correlations.csv`

In [ ]:
%%R
# ==============================================================================
# scripts/08_crossmodal_network_igraph.R
# Cultural Resistance: Counter-Visuality in the PCAP Archive
# PURPOSE: Computes the topic correlation matrix (stm::topicCorr) across the
#          25 LLaVA visual motifs and integrates CLIP visual clusters and
#          carceral material medium covariates into a cross-modal igraph
#          network visualization, testing how material scarcity (found/contraband
#          substrates) structurally links to counter-visual themes.
# OUTPUTS: data/pcap_visual_topic_corr.rds
#          results/figures/fig05_crossmodal_network_igraph.png
#          results/charts/table05_material_motif_correlations.csv
# ==============================================================================

drive_r_lib <- "/content/drive/MyDrive/colab_pcap_folder/r_libs"
if (dir.exists(drive_r_lib)) {
  .libPaths(unique(c(drive_r_lib, .libPaths())))
}

suppressPackageStartupMessages({
  library(tidyverse)
  library(stm)
  library(igraph)
})

cat("==============================================================================\n")
cat("[INFO] SCRIPT 08: Cross-Modal Network Topology (LLaVA Motifs x Mediums x CLIP)\n")
cat("==============================================================================\n")

if (dir.exists("/content/drive/MyDrive/colab_pcap_folder")) {
  setwd("/content/drive/MyDrive/colab_pcap_folder")
}

stm_k25 <- readRDS("data/pcap_stm_visual_k25.rds")
out_visual <- readRDS("data/pcap_llava_stm_out.rds")
table03_topics <- attr(stm_k25, "topic_summary_table")

# ------------------------------------------------------------------------------
# 1. COMPUTE STM TOPIC CORRELATION MATRIX (stm::topicCorr)
# ------------------------------------------------------------------------------
cat("[INFO] Computing stm::topicCorr across all 25 visual motifs...\n")
topic_corr_obj <- stm::topicCorr(stm_k25, method = "simple", cutoff = 0.05)
saveRDS(topic_corr_obj, "data/pcap_visual_topic_corr.rds")

# ------------------------------------------------------------------------------
# 2. COMPUTE CROSS-MODAL CORRELATIONS (MOTIFS x MEDIUMS x CLIP CLUSTERS)
# ------------------------------------------------------------------------------
theta_df <- as.data.frame(stm_k25$theta)
colnames(theta_df) <- sprintf("Topic_%02d", 1:25)

meta_df <- out_visual$meta %>% as_tibble()

medium_levels <- unique(as.character(meta_df$medium_category))
cluster_levels <- unique(as.character(meta_df$visual_cluster))

medium_mat <- sapply(medium_levels, function(m) as.numeric(meta_df$medium_category == m))
if (!is.matrix(medium_mat)) medium_mat <- matrix(medium_mat, ncol = length(medium_levels))
colnames(medium_mat) <- paste0("Medium: ", medium_levels)

cluster_mat <- sapply(cluster_levels, function(cl) as.numeric(meta_df$visual_cluster == cl))
if (!is.matrix(cluster_mat)) cluster_mat <- matrix(cluster_mat, ncol = length(cluster_levels))
colnames(cluster_mat) <- paste0("CLIP ", cluster_levels)

formal_cols <- intersect(
  c("formal_brightness", "formal_saturation", "formal_warm_ratio"),
  colnames(meta_df)
)
if (length(formal_cols) > 0) {
  formal_mat <- as.matrix(meta_df[, formal_cols, drop = FALSE])
  colnames(formal_mat) <- c(
    "Formal: Brightness (Intensity)",
    "Formal: Saturation (Chroma)",
    "Formal: Warm Color Ratio"
  )[seq_along(formal_cols)]
  cov_mat <- cbind(medium_mat, cluster_mat, formal_mat)
} else {
  cov_mat <- cbind(medium_mat, cluster_mat)
}

cross_corr_edges <- list()

for (k in 1:25) {
  t_vec <- theta_df[[k]]
  t_info <- table03_topics %>% filter(topic_id == k)
  t_short <- sprintf("T%02d (%s)", k, str_extract(t_info$frex_top10, "^[^,]+"))

  for (c_idx in seq_len(ncol(cov_mat))) {
    cov_name <- colnames(cov_mat)[c_idx]
    c_vec <- cov_mat[, c_idx]
    if (sd(t_vec, na.rm = TRUE) > 0 && sd(c_vec, na.rm = TRUE) > 0) {
      ct <- suppressWarnings(cor.test(t_vec, c_vec))
      cross_corr_edges[[length(cross_corr_edges) + 1]] <- tibble(
        from_node          = t_short,
        to_node            = cov_name,
        topic_id           = k,
        theoretical_family = t_info$theoretical_family,
        covariate_type     = case_when(
          str_starts(cov_name, "Medium:") ~ "Material Medium",
          str_starts(cov_name, "Formal:") ~ "Formal HCI Color/Light",
          TRUE                            ~ "CLIP Cluster"
        ),
        pearson_r          = round(as.numeric(ct$estimate), 4),
        p_value            = signif(ct$p.value, 4)
      )
    }
  }
}

table05_correlations <- bind_rows(cross_corr_edges) %>%
  arrange(desc(abs(pearson_r)))

readr::write_csv(table05_correlations, "results/charts/table05_material_motif_correlations.csv")
cat("[TDD PASS] Saved results/charts/table05_material_motif_correlations.csv\n")

# ------------------------------------------------------------------------------
# 3. BUILD & RENDER MULTIPARTITE IGRAPH NETWORK (FIGURE 05)
# ------------------------------------------------------------------------------
corr_mat <- topic_corr_obj$cor
topic_node_labels <- vapply(1:25, function(k) {
  t_info <- table03_topics %>% filter(topic_id == k)
  sprintf("T%02d (%s)", k, str_extract(t_info$frex_top10, "^[^,]+"))
}, character(1))

dimnames(corr_mat) <- list(Var1 = topic_node_labels, Var2 = topic_node_labels)

topic_edges <- as.data.frame(as.table(corr_mat), stringsAsFactors = FALSE) %>%
  setNames(c("Var1", "Var2", "n")) %>%
  as_tibble() %>%
  transmute(
    from_node = as.character(Var1),
    to_node   = as.character(Var2),
    weight    = as.numeric(n)
  ) %>%
  filter(from_node < to_node, weight > 0.06) %>%
  mutate(edge_type = "Topic-Topic Covariance")

cross_edges_filtered <- table05_correlations %>%
  filter(pearson_r > 0.03) %>%
  slice_head(n = 35) %>%
  transmute(
    from_node = as.character(from_node),
    to_node   = as.character(to_node),
    weight    = abs(pearson_r),
    edge_type = "Cross-Modal Link"
  )

all_edges <- bind_rows(topic_edges, cross_edges_filtered)
if (nrow(all_edges) == 0) {
  all_edges <- table05_correlations %>%
    slice_head(n = 20) %>%
    transmute(
      from_node = as.character(from_node),
      to_node   = as.character(to_node),
      weight    = pmax(abs(pearson_r), 0.05),
      edge_type = "Cross-Modal Link"
    )
}

g <- igraph::graph_from_data_frame(all_edges, directed = FALSE)

node_names <- V(g)$name
V(g)$modality <- case_when(
  str_starts(node_names, "Medium:") ~ "Carceral Material Medium",
  str_starts(node_names, "CLIP ")   ~ "CLIP Macro-Visual Cluster",
  TRUE                              ~ "LLaVA Iconographic Motif (STM)"
)

V(g)$color <- case_when(
  V(g)$modality == "Carceral Material Medium"       ~ "#B22222",
  V(g)$modality == "CLIP Macro-Visual Cluster"      ~ "#1B9E77",
  TRUE                                              ~ "#386CB0"
)

V(g)$size <- case_when(
  V(g)$modality == "Carceral Material Medium"       ~ 13,
  V(g)$modality == "CLIP Macro-Visual Cluster"      ~ 11,
  TRUE                                              ~ 8.5
)

set.seed(42)
layout_fr <- igraph::layout_with_fr(g, niter = 1000)

png(
  filename = "results/figures/fig05_crossmodal_network_igraph.png",
  width = 3200,
  height = 2400,
  res = 300,
  bg = "white"
)
par(mar = c(2, 2, 4, 2))
plot(
  g,
  layout = layout_fr,
  vertex.color = V(g)$color,
  vertex.frame.color = "white",
  vertex.size = V(g)$size,
  vertex.label.color = "grey10",
  vertex.label.cex = 0.72,
  vertex.label.dist = 0.65,
  vertex.label.family = "sans",
  edge.width = scales::rescale(E(g)$weight, to = c(1.0, 4.5)),
  edge.color = ifelse(E(g)$edge_type == "Cross-Modal Link", "#D95F0288", "#88888866"),
  main = "Cross-Modal Counter-Visuality Network: LLaVA Motifs, Carceral Mediums & CLIP Clusters"
)
legend(
  "bottomleft",
  legend = c(
    "Carceral Material Medium (Section 5.1)",
    "CLIP Visual Cluster (Section 5.2)",
    "LLaVA STM Visual Motif (Section 5.3)"
  ),
  pt.bg  = c("#B22222", "#1B9E77", "#386CB0"),
  pch    = 21,
  pt.cex = 2.0,
  cex    = 0.85,
  bty    = "n"
)
dev.off()

stopifnot(
  file.exists("data/pcap_visual_topic_corr.rds"),
  file.exists("results/figures/fig05_crossmodal_network_igraph.png"),
  file.exists("results/charts/table05_material_motif_correlations.csv")
)
cat("[TDD PASS] Script 08 Complete! All 3 target artifacts verified.\n")

[INFO] SCRIPT 08: Cross-Modal Network Topology (LLaVA Motifs x Mediums x CLIP)
[INFO] Computing stm::topicCorr across all 25 visual motifs...
[TDD PASS] Saved results/charts/table05_material_motif_correlations.csv
[TDD PASS] Script 08 Complete! All 3 target artifacts verified.


---
## Step 09: Qualitative Artwork Dossier & Joint CLIP–LLaVA Exemplar Extraction (`09_qualitative_artwork_dossier.R`)
Executes `stm::findThoughts()` combined with joint CLIP–LLaVA composite Z-score ranking to extract the top 5 exemplar artworks for each of the 5 target counter-visual motif families, writing a structured qualitative dossier for close art-historical reading.

**Output:** `results/charts/qualitative_artwork_dossier.txt`

In [ ]:
%%R
# ==============================================================================
# scripts/09_qualitative_artwork_dossier.R
# Cultural Resistance: Counter-Visuality in the PCAP Archive
# PURPOSE: Executes stm::findThoughts() combined with joint CLIP-LLaVA composite
#          Z-score ranking to extract the top 5 exemplar artworks for each of
#          the 5 target counter-visual motifs. Generates a formatted qualitative
#          dossier containing PCAP Item ID, Title, Exhibition Year, Medium,
#          Security Level, Artist Statement, LLaVA Iconographic Transcript, and
#          Omeka Web URL for close art-historical reading in Section 5.4.
# OUTPUT : results/charts/qualitative_artwork_dossier.txt
# ==============================================================================

suppressPackageStartupMessages({
  library(tidyverse)
  library(stm)
})

cat("==============================================================================\n")
cat("[INFO] SCRIPT 09: Joint CLIP-LLaVA Exemplar Extraction & Qualitative Dossier\n")
cat("==============================================================================\n")

if (dir.exists("/content/drive/MyDrive/colab_pcap_folder")) {
  setwd("/content/drive/MyDrive/colab_pcap_folder")
}

stm_k25 <- readRDS("data/pcap_stm_visual_k25.rds")
out_visual <- readRDS("data/pcap_llava_stm_out.rds")
table03_topics <- attr(stm_k25, "topic_summary_table")

meta_df <- out_visual$meta %>% as_tibble()
theta_mat <- stm_k25$theta

# ------------------------------------------------------------------------------
# 1. DEFINE TARGET COUNTER-VISUAL MOTIF FAMILIES & JOINT CLIP-LLAVA SCORING
# ------------------------------------------------------------------------------
target_families <- list(
  list(
    family_name = "Panoptic Eyes & Watchtowers",
    clip_col    = "score_surveillance_biometric",
    fallback_k  = 1L
  ),
  list(
    family_name = "Forensic Numbers & Barcodes",
    clip_col    = "score_state_bureaucracy",
    fallback_k  = 2L
  ),
  list(
    family_name = "Fragmented/Shackled Bodies & Counter-Gaze",
    clip_col    = "score_bodily_autonomy",
    fallback_k  = 3L
  ),
  list(
    family_name = "Pastoral Escape & Tactile Disalienation",
    clip_col    = "score_tactile_disalienation",
    fallback_k  = 4L
  ),
  list(
    family_name = "Telephonic/Screen Separation & Digital Mediation",
    clip_col    = "score_digital_mediation",
    fallback_k  = 5L
  )
)

dossier_path <- "results/charts/qualitative_artwork_dossier.txt"
con <- file(dossier_path, open = "wt", encoding = "UTF-8")

writeLines("================================================================================", con)
writeLines("QUALITATIVE ARTWORK DOSSIER — CHAPTER 5: COUNTER-VISUALITY IN THE PCAP ARCHIVE", con)
writeLines(sprintf("Generated: %s | Joint CLIP (ViT-L/14) + LLaVA (STM K=25) Ranking", Sys.time()), con)
writeLines("================================================================================\n", con)

for (fam in target_families) {
  fam_name <- fam$family_name
  clip_col <- fam$clip_col

  matched_topics <- table03_topics %>%
    filter(theoretical_family == fam_name) %>%
    arrange(desc(anchor_overlap), desc(expected_prop))

  if (nrow(matched_topics) > 0) {
    primary_k <- matched_topics$topic_id[1]
    frex_words <- matched_topics$frex_top10[1]
  } else {
    primary_k <- min(fam$fallback_k, ncol(theta_mat))
    frex_words <- table03_topics %>% filter(topic_id == primary_k) %>% pull(frex_top10)
  }

  # Execute stm::findThoughts() on the primary motif topic to verify STM alignment
  n_thoughts <- min(15L, nrow(meta_df))
  ft_obj <- stm::findThoughts(
    model  = stm_k25,
    texts  = meta_df$llava_description,
    topics = primary_k,
    n      = n_thoughts
  )
  ft_indices <- unlist(ft_obj$index)

  # Compute Joint Composite Score = 0.5 * Z(STM Theta_k) + 0.5 * Z(CLIP Cosine Similarity)
  # For Family 3 (Fragmented/Shackled Bodies & Counter-Gaze), combine Bodily Autonomy
  # and Carceral Containment so works depicting human subjects in chains/cells rank highest.
  stm_theta_vec <- theta_mat[, primary_k]
  if (fam_name == "Fragmented/Shackled Bodies & Counter-Gaze") {
    clip_score_vec <- 0.5 * meta_df$score_bodily_autonomy + 0.5 * meta_df$score_carceral_containment
  } else {
    clip_score_vec <- meta_df[[clip_col]]
  }

  z_stm  <- as.numeric(scale(stm_theta_vec))
  z_clip <- as.numeric(scale(clip_score_vec))
  joint_score <- 0.5 * coalesce(z_stm, 0) + 0.5 * coalesce(z_clip, 0)

  exemplars_df <- meta_df %>%
    mutate(
      doc_idx         = row_number(),
      stm_theta       = round(stm_theta_vec, 4),
      clip_sim        = round(clip_score_vec, 4),
      joint_z         = round(joint_score, 4),
      in_findthoughts = doc_idx %in% ft_indices
    ) %>%
    arrange(desc(in_findthoughts), desc(joint_z)) %>%
    slice_head(n = 5)

  writeLines("--------------------------------------------------------------------------------", con)
  writeLines(sprintf("MOTIF FAMILY : %s", toupper(fam_name)), con)
  writeLines(sprintf("PRIMARY STM  : Topic %02d | Top FREX Terms: %s", primary_k, frex_words), con)
  writeLines(sprintf("CLIP AXIS    : %s", clip_col), con)
  writeLines("--------------------------------------------------------------------------------\n", con)

  for (r in seq_len(nrow(exemplars_df))) {
    ex <- exemplars_df[r, ]
    stmt_str <- ifelse(
      is.na(ex$artist_statement) || nchar(str_trim(ex$artist_statement)) == 0,
      "[No artist statement archived on Omeka S item record]",
      ex$artist_statement
    )

    writeLines(sprintf("  [EXEMPLAR #%d | Joint Z-Score: %+.3f | STM Theta: %.3f | CLIP Sim: %.3f]",
                       r, ex$joint_z, ex$stm_theta, ex$clip_sim), con)
    writeLines(sprintf("  PCAP Item ID     : %s (Log/ID: %s)", ex$item_id, coalesce(ex$identifier, "N/A")), con)
    writeLines(sprintf("  Artwork Title    : %s", coalesce(ex$title, "Untitled")), con)
    writeLines(sprintf("  Artist           : %s", coalesce(ex$artist_name, "Anonymous")), con)
    writeLines(sprintf("  Exhibition Year  : %s (%s)", as.character(ex$clean_year), coalesce(ex$exhibition_source, "PCAP Annual Exhibition")), con)
    writeLines(sprintf("  Material Medium  : %s [Taxonomy: %s]", coalesce(ex$medium_raw, "Unspecified"), as.character(ex$medium_category)), con)
    writeLines(sprintf("  Facility / Level : %s [%s]", coalesce(ex$facility_raw, "Unspecified"), coalesce(ex$security_level, "Unspecified")), con)
    writeLines(sprintf("  Omeka S Web URL  : %s", coalesce(ex$omeka_web_url, "N/A")), con)
    writeLines(sprintf("  Artist Statement : %s", stmt_str), con)
    writeLines(sprintf("  LLaVA Transcript : %s\n", ex$llava_description), con)
  }
}

close(con)

stopifnot(
  file.exists(dossier_path),
  file.info(dossier_path)$size > 200
)
cat(sprintf("[TDD PASS] Script 09 Complete! Qualitative dossier written to %s (%d bytes).\n",
            dossier_path, file.info(dossier_path)$size))
writeLines(capture.output(sessionInfo()), "results/session_info.txt")
cat("[INFO] Exported R sessionInfo() to results/session_info.txt.\n")

[INFO] SCRIPT 09: Joint CLIP-LLaVA Exemplar Extraction & Qualitative Dossier
[TDD PASS] Script 09 Complete! Qualitative dossier written to results/charts/qualitative_artwork_dossier.txt (34307 bytes).
[INFO] Exported R sessionInfo() to results/session_info.txt.
